In [5]:


import pandas as pd
import numpy as np

# Load support pairs
support_pairs_df = pd.read_csv(
    "../data/processed/support_pairs.csv"
)

# Load semantic cluster labels
semantic_cluster_labels = np.load(
    "../data/processed/semantic_cluster_labels.npy"
)

print("Support pairs:", len(support_pairs_df))
print("Semantic labels:", len(semantic_cluster_labels))

print("\nColumns:")
print(support_pairs_df.columns.tolist())

print("\nSemantic cluster counts:")
print(
    pd.Series(
        semantic_cluster_labels
    ).value_counts().sort_index()
)

Support pairs: 3092
Semantic labels: 3092

Columns:
['conversation_id', 'customer_tweet_id', 'support_tweet_id', 'customer_message', 'support_response', 'response_behavior', 'response_quality']

Semantic cluster counts:
0     311
1     381
2     111
3     196
4     293
5      78
6     361
7     180
8     295
9     409
10    161
11    316
Name: count, dtype: int64


In [6]:
# Cell 2 — Compare semantic clusters with support response behavior

validation_df = support_pairs_df.copy()

validation_df["semantic_cluster"] = semantic_cluster_labels

cluster_behavior = pd.crosstab(
    validation_df["semantic_cluster"],
    validation_df["response_behavior"],
    normalize="index"
).round(3)

print("Response behavior distribution within each semantic cluster:")
display(cluster_behavior)

print("\nRaw counts:")

cluster_behavior_counts = pd.crosstab(
    validation_df["semantic_cluster"],
    validation_df["response_behavior"]
)

display(cluster_behavior_counts)

Response behavior distribution within each semantic cluster:


response_behavior,acknowledgement,dm_escalation,information_request,link_resource,other,troubleshooting
semantic_cluster,,,,,,
0,0.006,0.302,0.235,0.055,0.029,0.373
1,0.010,0.352,0.129,0.102,0.110,0.297
2,0.000,0.198,0.144,0.027,0.036,0.595
3,0.005,0.388,0.224,0.005,0.000,0.378
4,0.007,0.242,0.191,0.157,0.038,0.365
5,0.013,0.000,0.000,0.987,0.000,0.000
6,0.006,0.302,0.197,0.058,0.097,0.341
7,0.011,0.406,0.178,0.011,0.028,0.367
8,0.010,0.369,0.251,0.054,0.014,0.302



Raw counts:


response_behavior,acknowledgement,dm_escalation,information_request,link_resource,other,troubleshooting
semantic_cluster,,,,,,
0,2,94,73,17,9,116
1,4,134,49,39,42,113
2,0,22,16,3,4,66
3,1,76,44,1,0,74
4,2,71,56,46,11,107
5,1,0,0,77,0,0
6,2,109,71,21,35,123
7,2,73,32,2,5,66
8,3,109,74,16,4,89


In [7]:
# Cell 3 — Define candidate business-oriented intent taxonomy

candidate_intents = {
    "software_update": (
        "Problems caused by iOS/software updates, "
        "update installation, or update-related behavior"
    ),

    "keyboard_input": (
        "Keyboard, typing, autocorrect, symbols, "
        "or text input problems"
    ),

    "battery_power": (
        "Battery drain, battery life, charging, "
        "or power-related problems"
    ),

    "performance_freezing": (
        "Phone/Mac performance issues such as lagging, "
        "freezing, crashing, or becoming unresponsive"
    ),

    "apps_app_store": (
        "Apps, App Store, downloading, installing, "
        "or opening applications"
    ),

    "apple_music": (
        "Apple Music, songs, playlists, downloads, "
        "or music playback problems"
    ),

    "apple_id_icloud": (
        "Apple ID, iCloud, account access, password, "
        "verification, or account lock issues"
    ),

    "device_hardware": (
        "Physical device problems, hardware failure, "
        "unexpected shutdowns, or device replacement issues"
    ),

    "connectivity": (
        "Wi-Fi, Bluetooth, cellular, or other "
        "device connectivity problems"
    ),

    "other_unclear": (
        "Messages that do not contain enough information "
        "to confidently assign a specific support intent"
    )
}

print("Candidate intent taxonomy:")
print("-" * 80)

for i, (intent, description) in enumerate(
    candidate_intents.items(),
    start=1
):
    print(f"{i}. {intent}")
    print(f"   {description}")

print("\nTotal candidate intents:", len(candidate_intents))

Candidate intent taxonomy:
--------------------------------------------------------------------------------
1. software_update
   Problems caused by iOS/software updates, update installation, or update-related behavior
2. keyboard_input
   Keyboard, typing, autocorrect, symbols, or text input problems
3. battery_power
   Battery drain, battery life, charging, or power-related problems
4. performance_freezing
   Phone/Mac performance issues such as lagging, freezing, crashing, or becoming unresponsive
5. apps_app_store
   Apps, App Store, downloading, installing, or opening applications
6. apple_music
   Apple Music, songs, playlists, downloads, or music playback problems
7. apple_id_icloud
   Apple ID, iCloud, account access, password, verification, or account lock issues
8. device_hardware
   Physical device problems, hardware failure, unexpected shutdowns, or device replacement issues
9. connectivity
   Wi-Fi, Bluetooth, cellular, or other device connectivity problems
10. other_uncle

In [10]:
# Cell 4 — Check whether candidate intents are represented
# by the discovered semantic clusters

n_clusters = 12

cluster_intent_evidence = {
    0: ["software_update"],
    1: ["other_unclear"],
    2: ["software_update"],
    3: ["keyboard_input"],
    4: ["apps_app_store"],
    5: [
        "software_update",
        "battery_power",
        "performance_freezing",
        "connectivity"
    ],
    6: [
        "performance_freezing",
        "other_unclear"
    ],
    7: ["battery_power"],
    8: [
        "software_update",
        "performance_freezing"
    ],
    9: [
        "performance_freezing",
        "device_hardware",
        "software_update"
    ],
    10: ["apple_music"],
    11: ["apple_id_icloud"]
}

print("Candidate intent evidence from semantic clusters:\n")

for cluster_id, intents in cluster_intent_evidence.items():

    cluster_size = (
        validation_df["semantic_cluster"]
        == cluster_id
    ).sum()

    print(
        f"Cluster {cluster_id:2d} "
        f"(size={cluster_size:3d})"
    )

    print(
        "  Possible intents:",
        ", ".join(intents)
    )

print("\nCandidate intents:", len(candidate_intents))
print("Semantic clusters:", n_clusters)

Candidate intent evidence from semantic clusters:

Cluster  0 (size=311)
  Possible intents: software_update
Cluster  1 (size=381)
  Possible intents: other_unclear
Cluster  2 (size=111)
  Possible intents: software_update
Cluster  3 (size=196)
  Possible intents: keyboard_input
Cluster  4 (size=293)
  Possible intents: apps_app_store
Cluster  5 (size= 78)
  Possible intents: software_update, battery_power, performance_freezing, connectivity
Cluster  6 (size=361)
  Possible intents: performance_freezing, other_unclear
Cluster  7 (size=180)
  Possible intents: battery_power
Cluster  8 (size=295)
  Possible intents: software_update, performance_freezing
Cluster  9 (size=409)
  Possible intents: performance_freezing, device_hardware, software_update
Cluster 10 (size=161)
  Possible intents: apple_music
Cluster 11 (size=316)
  Possible intents: apple_id_icloud

Candidate intents: 10
Semantic clusters: 12


In [11]:
# Cell 5 — Estimate coverage of the candidate taxonomy
#
# This is an exploratory estimate based on semantic-cluster mapping.
# It is NOT the final ground-truth labeling.

cluster_sizes = (
    validation_df["semantic_cluster"]
    .value_counts()
    .sort_index()
)

intent_coverage = {}

for intent in candidate_intents:

    covered_clusters = [
        cluster_id
        for cluster_id, intents in cluster_intent_evidence.items()
        if intent in intents
    ]

    coverage_count = sum(
        cluster_sizes.get(cluster_id, 0)
        for cluster_id in covered_clusters
    )

    coverage_percentage = (
        coverage_count / len(validation_df) * 100
    )

    intent_coverage[intent] = {
        "clusters": covered_clusters,
        "messages": coverage_count,
        "coverage_percent": round(
            coverage_percentage, 2
        )
    }

coverage_df = pd.DataFrame(
    intent_coverage
).T.reset_index()

coverage_df = coverage_df.rename(
    columns={"index": "intent"}
)

print("Estimated candidate intent coverage:")
display(coverage_df)

Estimated candidate intent coverage:


,intent,clusters,messages,coverage_percent
0,software_update,"[0, 2, 5, 8, 9]",1204,38.94
1,keyboard_input,[3],196,6.34
2,battery_power,"[5, 7]",258,8.34
3,performance_freezing,"[5, 6, 8, 9]",1143,36.97
4,apps_app_store,[4],293,9.48
5,apple_music,[10],161,5.21
6,apple_id_icloud,[11],316,10.22
7,device_hardware,[9],409,13.23
8,connectivity,[5],78,2.52
9,other_unclear,"[1, 6]",742,24.0


In [12]:
# Cell 6 — Sample real messages for candidate taxonomy validation

import numpy as np

np.random.seed(42)

for intent, evidence in intent_coverage.items():

    covered_clusters = evidence["clusters"]

    candidate_rows = validation_df[
        validation_df["semantic_cluster"].isin(
            covered_clusters
        )
    ]

    sample_size = min(5, len(candidate_rows))

    samples = candidate_rows.sample(
        n=sample_size,
        random_state=42
    )

    print("=" * 90)
    print(f"INTENT: {intent}")
    print(
        f"Evidence clusters: {covered_clusters}"
    )
    print(
        f"Candidate messages available: "
        f"{len(candidate_rows)}"
    )
    print("=" * 90)

    for i, (_, row) in enumerate(
        samples.iterrows(),
        start=1
    ):
        print(f"{i}. {row['customer_message']}")

    print()

INTENT: software_update
Evidence clusters: [0, 2, 5, 8, 9]
Candidate messages available: 1204
1. Hi, any thoughts on why my iPhone is suddenly making a crackling sound when I refresh Twitter, as an example. Sounds vaguely like an electrical interference? Thanks.
2. Tried to update my phone to the new IOS system, and my phone died while 'charging' &amp; now I lost everything. Thanks @115858.
3. It’s 11.0.3
4. @115858  you wana tell me why my brand new iPhone 8 is lagging already? I already updated to the newest IOS.
5. @115858 Good to see we have time for commercials but can't get any help with fixing the new iOS update that is draining my battery.....

INTENT: keyboard_input
Evidence clusters: [3]
Candidate messages available: 196
1. @115858 creates amazing technology..... but talk to text still messes up 90% of what I will it to type..... #fixyourcrap
2. Anybody else experiencing the bug where the letter I turning into a weird set of glyphs? Igate
3. Man I’m having the same problem. T

In [13]:
# Cell 7 — Define intent labeling guidelines
#
# These rules will be used to create consistent provisional labels.
# A message should be assigned based on the customer's MAIN problem,
# not merely because an intent-related word appears in the message.

intent_guidelines = {
    "software_update": [
        "The main issue is installing, updating, or reverting iOS/macOS/software.",
        "The customer explicitly connects the problem to a software update.",
        "Example: 'After updating to iOS 11 my phone became slow.'"
    ],

    "keyboard_input": [
        "The problem involves typing, keyboard behavior, autocorrect, text input, or symbols.",
        "Example: 'The letter I keeps changing into a strange symbol.'"
    ],

    "battery_power": [
        "The main issue is battery drain, charging, battery life, or power loss.",
        "Example: 'My battery loses 50% in one hour.'"
    ],

    "performance_freezing": [
        "The main issue is lag, freezing, crashing, slowness, or an unresponsive device.",
        "Use this when performance is the main problem even if no update is mentioned."
    ],

    "apps_app_store": [
        "The main issue involves an app, App Store, app installation, downloading, or opening an app.",
        "Example: 'The App Store will not download my applications.'"
    ],

    "apple_music": [
        "The main issue specifically involves Apple Music, songs, playlists, music playback, or Apple Music downloads.",
        "Do NOT use this merely because music is mentioned."
    ],

    "apple_id_icloud": [
        "The main issue involves Apple ID, iCloud, password, account access, verification, or account lock.",
        "The account problem must be the customer's main issue."
    ],

    "device_hardware": [
        "The main issue appears to be physical hardware failure or a device-level hardware problem.",
        "Examples include broken hardware, display damage, hardware failure, or device replacement concerns.",
        "Do NOT use this for normal software bugs."
    ],

    "connectivity": [
        "The main issue involves Wi-Fi, Bluetooth, cellular/mobile network, AirDrop, or another connection problem."
    ],

    "other_unclear": [
        "The message is too vague to confidently assign another intent.",
        "Use for acknowledgements, thanks, incomplete context, or genuinely unrelated/ambiguous requests."
    ]
}

print("Intent labeling guidelines:\n")

for i, (intent, rules) in enumerate(
    intent_guidelines.items(),
    start=1
):
    print(f"{i}. {intent}")

    for rule in rules:
        print(f"   - {rule}")

    print()

print(
    "Total intents:",
    len(intent_guidelines)
)

Intent labeling guidelines:

1. software_update
   - The main issue is installing, updating, or reverting iOS/macOS/software.
   - The customer explicitly connects the problem to a software update.
   - Example: 'After updating to iOS 11 my phone became slow.'

2. keyboard_input
   - The problem involves typing, keyboard behavior, autocorrect, text input, or symbols.
   - Example: 'The letter I keeps changing into a strange symbol.'

3. battery_power
   - The main issue is battery drain, charging, battery life, or power loss.
   - Example: 'My battery loses 50% in one hour.'

4. performance_freezing
   - The main issue is lag, freezing, crashing, slowness, or an unresponsive device.
   - Use this when performance is the main problem even if no update is mentioned.

5. apps_app_store
   - The main issue involves an app, App Store, app installation, downloading, or opening an app.
   - Example: 'The App Store will not download my applications.'

6. apple_music
   - The main issue specifi

In [14]:
# Cell 8 — Create provisional intent labels using explicit rules

import re

def assign_provisional_intent(text):
    text = str(text).lower().strip()

    if not text:
        return "other_unclear"

    # Remove URLs and user mentions
    cleaned = re.sub(r"https?://\S+|www\.\S+", " ", text)
    cleaned = re.sub(r"@\w+", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # ---------------------------------------------------------
    # 1. Keyboard / typing
    # ---------------------------------------------------------
    keyboard_terms = [
        "keyboard",
        "autocorrect",
        "auto correct",
        "typing",
        "type",
        "typed",
        "typing",
        "text input",
        "letter i",
        "letter \"i\"",
        "symbols",
        "symbol",
        "glyph"
    ]

    if any(term in cleaned for term in keyboard_terms):
        return "keyboard_input"

    # ---------------------------------------------------------
    # 2. Battery / power
    # ---------------------------------------------------------
    battery_terms = [
        "battery",
        "battery life",
        "battery drain",
        "battery draining",
        "draining battery",
        "charge",
        "charging",
        "power drain",
        "losing battery",
        "loses battery"
    ]

    if any(term in cleaned for term in battery_terms):
        return "battery_power"

    # ---------------------------------------------------------
    # 3. Apple Music
    # ---------------------------------------------------------
    music_terms = [
        "apple music",
        "itunes",
        "music app",
        "music library",
        "playlist",
        "playlists",
        "song won't play",
        "songs won't play",
        "music won't play"
    ]

    if any(term in cleaned for term in music_terms):
        return "apple_music"

    # ---------------------------------------------------------
    # 4. Apple ID / iCloud
    # ---------------------------------------------------------
    account_terms = [
        "apple id",
        "appleid",
        "icloud",
        "i cloud",
        "account locked",
        "account lock",
        "forgot password",
        "forgot my password",
        "verification code",
        "two factor",
        "2 factor",
        "2fa",
        "sign in",
        "login",
        "log in"
    ]

    if any(term in cleaned for term in account_terms):
        return "apple_id_icloud"

    # ---------------------------------------------------------
    # 5. Connectivity
    # ---------------------------------------------------------
    connectivity_terms = [
        "wifi",
        "wi-fi",
        "wi fi",
        "bluetooth",
        "cellular",
        "mobile network",
        "network",
        "airdrop",
        "air drop",
        "no signal",
        "signal",
        "internet connection",
        "connection"
    ]

    if any(term in cleaned for term in connectivity_terms):
        return "connectivity"

    # ---------------------------------------------------------
    # 6. Apps / App Store
    # ---------------------------------------------------------
    app_terms = [
        "app store",
        "appstore",
        "apps",
        "app",
        "application",
        "applications",
        "download app",
        "install app",
        "app won't open",
        "app won't work",
        "app not working",
        "application not working"
    ]

    if any(term in cleaned for term in app_terms):
        return "apps_app_store"

    # ---------------------------------------------------------
    # 7. Software update
    # ---------------------------------------------------------
    update_terms = [
        "ios update",
        "ios updates",
        "ios 11",
        "ios11",
        "updated ios",
        "update ios",
        "software update",
        "software updates",
        "latest update",
        "new update",
        "recent update",
        "after update",
        "after updating",
        "upgraded to",
        "upgrade",
        "downgrade",
        "revert update"
    ]

    if any(term in cleaned for term in update_terms):
        return "software_update"

    # ---------------------------------------------------------
    # 8. Performance / freezing
    # ---------------------------------------------------------
    performance_terms = [
        "lag",
        "lagging",
        "slow",
        "slower",
        "slowly",
        "freeze",
        "freezing",
        "frozen",
        "crash",
        "crashing",
        "crashed",
        "unresponsive",
        "not responding",
        "stuck",
        "glitch",
        "glitching",
        "hang",
        "hanging"
    ]

    if any(term in cleaned for term in performance_terms):
        return "performance_freezing"

    # ---------------------------------------------------------
    # 9. Device / hardware
    # ---------------------------------------------------------
    hardware_terms = [
        "broken screen",
        "cracked screen",
        "physical damage",
        "hardware",
        "screen damage",
        "display damage",
        "won't turn on",
        "doesn't turn on",
        "not turning on",
        "device replacement",
        "replace my iphone",
        "replace my phone"
    ]

    if any(term in cleaned for term in hardware_terms):
        return "device_hardware"

    # ---------------------------------------------------------
    # 10. Everything else
    # ---------------------------------------------------------
    return "other_unclear"


# Apply provisional labels
validation_df["provisional_intent"] = (
    validation_df["customer_message"]
    .apply(assign_provisional_intent)
)

print("Provisional intent distribution:")
print(
    validation_df["provisional_intent"]
    .value_counts()
)

print(
    "\nTotal messages:",
    len(validation_df)
)

Provisional intent distribution:
provisional_intent
other_unclear           1406
apps_app_store           534
battery_power            258
software_update          257
keyboard_input           178
performance_freezing     146
connectivity             115
apple_music              109
apple_id_icloud           84
device_hardware            5
Name: count, dtype: int64

Total messages: 3092


In [15]:
# Cell 9 — Inspect provisional "other_unclear" messages

unclear_samples = (
    validation_df[
        validation_df["provisional_intent"] == "other_unclear"
    ][
        ["customer_message", "support_response"]
    ]
    .sample(
        n=min(50, len(
            validation_df[
                validation_df["provisional_intent"] == "other_unclear"
            ]
        )),
        random_state=42
    )
    .reset_index(drop=True)
)

for i, row in unclear_samples.iterrows():
    print(f"\n--- Example {i + 1} ---")
    print("Customer:", row["customer_message"])
    print("Support :", row["support_response"])


--- Example 1 ---
Customer: I️ I️ I️ I️ I️ I️ I️ I️ I️ I️ 
Dear @115858 r y’all working on a fix for this??
Support : @134117 We're more than happy to help you fix this. Can you tell us which iOS version you are currently using please?

--- Example 2 ---
Customer: @AppleSupport  https://t.co/FUUKlavB1I
Support : @130266 Join us in DM by clicking on the link below. We'll continue with support in there. https://t.co/GDrqU22YpT

--- Example 3 ---
Customer: I I need y’all to fix this
Support : @125132 We're here to help. Let's take this to DM so we can better assist you. https://t.co/GDrqU22YpT

--- Example 4 ---
Customer: I’m turning it off from the control center I believe
Support : @122947 OK! Take a look here for the way Wi-Fi functions in the Control Center of iOS 11:  https://t.co/G7BHuQnDS6

--- Example 5 ---
Customer: The link didn’t really help me
Support : @136103 We're sorry about that. The link we provided has some information on what's advisable and what isn't. Generally, it 

In [17]:
# Cell 11 — Compare first and second provisional labeling rules

# Re-create the original rule labels
validation_df["provisional_intent_v1"] = (
    validation_df["customer_message"]
    .apply(assign_provisional_intent)
)

# v2 is already stored in provisional_intent
validation_df["provisional_intent_v2"] = (
    validation_df["customer_message"]
    .apply(assign_provisional_intent_v2)
)

comparison = pd.crosstab(
    validation_df["provisional_intent_v1"],
    validation_df["provisional_intent_v2"]
)

print("V1 → V2 label changes:\n")
print(comparison)

print("\nNumber of messages whose label changed:")

changed = (
    validation_df["provisional_intent_v1"]
    != validation_df["provisional_intent_v2"]
)

print(changed.sum())
print(
    "Percentage changed:",
    round(changed.mean() * 100, 2),
    "%"
)

V1 → V2 label changes:

provisional_intent_v2  apple_id_icloud  apple_music  apps_app_store  \
provisional_intent_v1                                                 
apple_id_icloud                     84            0               0   
apple_music                          0          109               0   
apps_app_store                       1            0              44   
battery_power                        1            6               0   
connectivity                         0            0               0   
device_hardware                      0            0               0   
keyboard_input                       1            1               0   
other_unclear                        6            0               3   
performance_freezing                 1            0               1   
software_update                      1            0               0   

provisional_intent_v2  battery_power  connectivity  device_hardware  \
provisional_intent_v1                               

In [19]:
# Cell 12 — Attach semantic cluster labels to validation_df

import numpy as np
import pandas as pd

# Load the semantic cluster labels created in Notebook 04
semantic_labels = np.load(
    "../data/processed/semantic_cluster_labels.npy"
)

print("Number of cluster labels:", len(semantic_labels))
print("Number of validation rows:", len(validation_df))

# Safety check
assert len(semantic_labels) == len(validation_df), (
    "Cluster labels and validation_df have different lengths."
)

# Attach labels
validation_df["semantic_cluster"] = semantic_labels

print("\nSemantic cluster column added successfully.")

print("\nCluster distribution:")
print(
    validation_df["semantic_cluster"]
    .value_counts()
    .sort_index()
)

Number of cluster labels: 3092
Number of validation rows: 3092

Semantic cluster column added successfully.

Cluster distribution:
semantic_cluster
0     311
1     381
2     111
3     196
4     293
5      78
6     361
7     180
8     295
9     409
10    161
11    316
Name: count, dtype: int64


In [21]:
# Cell 13 — Create balanced human-annotation sample
# Using a simple loop instead of groupby().apply()

import pandas as pd

samples = []

for cluster_id in sorted(validation_df["semantic_cluster"].unique()):

    cluster_rows = validation_df[
        validation_df["semantic_cluster"] == cluster_id
    ]

    n_samples = min(10, len(cluster_rows))

    sampled_rows = cluster_rows.sample(
        n=n_samples,
        random_state=42
    )

    samples.append(
        sampled_rows[
            [
                "customer_message",
                "support_response",
                "semantic_cluster"
            ]
        ]
    )

annotation_sample = pd.concat(
    samples,
    ignore_index=True
)

print("Annotation sample size:", len(annotation_sample))

print("\nMessages sampled from each semantic cluster:")
print(
    annotation_sample["semantic_cluster"]
    .value_counts()
    .sort_index()
)

print("\nFirst 20 examples:")

for i, row in annotation_sample.head(20).iterrows():

    print(f"\n--- Example {i + 1} ---")
    print("Cluster :", row["semantic_cluster"])
    print("Customer:", row["customer_message"])
    print("Support :", row["support_response"])

Annotation sample size: 120

Messages sampled from each semantic cluster:
semantic_cluster
0     10
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
Name: count, dtype: int64

First 20 examples:

--- Example 1 ---
Cluster : 0
Customer: Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg
Support : @136576 Hello. Thanks for bringing this to our attention. We'd like to team up with you and get this sorted. Click here to join us in DM and provide us with the device and iOS 11 version that you're using: https://t.co/GDrqU22YpT

--- Example 2 ---
Cluster : 0
Customer: I am using an iPhone 8 Plus and iOS 11.1.2
Support : @139262 Got it, thanks. While automatic playback is expected is your vehicles firmware up to date? There might be additional settings there you can adjust. Let us know in a DM and we'll continue. https://t.co/GDrqU

In [22]:
# Cell 14 — Prepare human annotation table

annotation_df = annotation_sample.copy()

# Add empty columns for manual annotation
annotation_df["human_intent"] = ""
annotation_df["annotation_confidence"] = ""
annotation_df["annotation_notes"] = ""

# Add a stable annotation ID
annotation_df.insert(
    0,
    "annotation_id",
    range(1, len(annotation_df) + 1)
)

print("Annotation dataframe created.")
print("\nShape:", annotation_df.shape)

print("\nColumns:")
print(annotation_df.columns.tolist())

print("\nAnnotation fields:")
print("- human_intent: final manually validated intent")
print("- annotation_confidence: high / medium / low")
print("- annotation_notes: short reason for the decision")

Annotation dataframe created.

Shape: (120, 7)

Columns:
['annotation_id', 'customer_message', 'support_response', 'semantic_cluster', 'human_intent', 'annotation_confidence', 'annotation_notes']

Annotation fields:
- human_intent: final manually validated intent
- annotation_confidence: high / medium / low
- annotation_notes: short reason for the decision


In [23]:
# Cell 15 — Display all 120 messages for manual annotation

pd.set_option("display.max_colwidth", 300)

annotation_view = annotation_df[
    [
        "annotation_id",
        "customer_message",
        "semantic_cluster",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
].copy()

display(annotation_view)

,annotation_id,customer_message,semantic_cluster,human_intent,annotation_confidence,annotation_notes
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,0,,,
1,2,I am using an iPhone 8 Plus and iOS 11.1.2,0,,,
2,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",0,,,
3,4,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,0,,,
4,5,"About 2 weeks, I have IOS 11.1.2",0,,,
...,...,...,...,...,...,...
115,116,Four videos are suddenly gone from my phone...would the automatic iCloud backing-up do that? Clear space by transferring them there?? That's the only thing that could make any sense.,11,,,
116,117,where do we forward fake Apple emails?,11,,,
117,118,Hey - Over the last few days my messages app on my iMac and MBA stopped syncing with my iMessages from my iOS devices. Any ideas on how to fix this problem? I've lost one of my favorite parts of AppleSimplicity,11,,,
118,119,I'm desperate. my grandkid just turned on voiceover with the internet off. I tried rebooting and now it wants my passcode but won't let me enter it. How do I get access to my ipad again.,11,,,


In [24]:
# Cell 16 — Finalize annotation IDs and save template

annotation_df["annotation_id"] = range(1, len(annotation_df) + 1)

annotation_df.to_csv(
    "../data/processed/human_annotation_template.csv",
    index=False
)

print("Annotation template saved.")
print("Shape:", annotation_df.shape)
print("ID range:", annotation_df["annotation_id"].min(), "to", annotation_df["annotation_id"].max())
print("File: ../data/processed/human_annotation_template.csv")

Annotation template saved.
Shape: (120, 7)
ID range: 1 to 120
File: ../data/processed/human_annotation_template.csv


In [25]:
# Cell 17 — Verify balanced annotation sample

cluster_summary = (
    annotation_df["semantic_cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("semantic_cluster")
    .reset_index(name="sample_count")
)

display(cluster_summary)

print("Total samples:", len(annotation_df))
print("Expected samples per cluster:", 10)
print(
    "Balanced:",
    cluster_summary["sample_count"].eq(10).all()
)

,semantic_cluster,sample_count
0,0,10
1,1,10
2,2,10
3,3,10
4,4,10
5,5,10
6,6,10
7,7,10
8,8,10
9,9,10


Total samples: 120
Expected samples per cluster: 10
Balanced: True


In [26]:
# Cell 18 — Inspect Cluster 0 messages

cluster_0 = annotation_df[
    annotation_df["semantic_cluster"] == 0
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_0)

,annotation_id,customer_message,support_response
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,@136576 Hello. Thanks for bringing this to our attention. We'd like to team up with you and get this sorted. Click here to join us in DM and provide us with the device and iOS 11 version that you're using: https://t.co/GDrqU22YpT
1,2,I am using an iPhone 8 Plus and iOS 11.1.2,"@139262 Got it, thanks. While automatic playback is expected is your vehicles firmware up to date? There might be additional settings there you can adjust. Let us know in a DM and we'll continue. https://t.co/GDrqU22YpT"
2,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?","@143276 We're happy to help with these issues. What happens when you attempt to open the photo gallery currently? Also, let's see if we can resolve that camera issue with the steps outlined here: https://t.co/r1DRfaFLIU"
3,4,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,@143869 We don't want you to be without any sound on your device. We can definitely look into this further with you. \nLet us know if this helps: https://t.co/4Kux8rhhGX
4,5,"About 2 weeks, I have IOS 11.1.2",@141071 Thanks for that info. Let's try a few things in this article to see if it helps. Try steps 4 through 6:\nhttps://t.co/UTP2qBi1Is
5,6,I got a crazy security bug issue with iOS 11.1,@118835 We'd like to help. Let's take this to DM and we'll explore ways to provide you assistance. https://t.co/GDrqU22YpT
6,7,force touch for app multitask gone in iOS11 ?,@121981 That feature isn’t a part of iOS 11. We always enjoy hearing feedback. Please submit yours here:\nhttps://t.co/oWzjJxkvlC
7,8,Hey iOS 11 sucks none of my apps can update anymore,"@127203 Let us know via DM what occurs when trying to update, then we can work together to get it resolved. https://t.co/GDrqU22YpT"
8,9,ios 11,@134149 Let's continue in DM. You can use the following link to get in touch with us directly: https://t.co/GDrqU22YpT
9,10,": New iPhoneX. iOS 11.1.2 update. Phone app literally will not open. Cannot access call history, my voicemail or make a call. WTF","@142502 That's not the experience we want you to have with your new iPhone X and we'd like to help. To start, DM us with the country where you're located. We look forward to working with you to resolve this. https://t.co/GDrqU22YpT"


In [27]:
# Cell 19 — Add human-validated labels for Cluster 0

cluster_0_labels = {
    1: ("keyboard_input", "high",
        "Delete key on Bluetooth keyboard stopped working while typing."),
    
    2: ("other_unclear", "high",
        "Only provides device and iOS version; no problem is stated."),
    
    3: ("performance_freezing", "high",
        "Phone repeatedly freezes; camera and gallery problems are secondary."),
    
    4: ("software_update", "high",
        "No sound started immediately after the iOS update."),
    
    5: ("other_unclear", "high",
        "Only states the iOS version and duration; no clear support issue."),
    
    6: ("software_update", "medium",
        "Security bug is associated with iOS 11.1, but the exact problem is unclear."),
    
    7: ("software_update", "high",
        "Customer reports a feature being unavailable in iOS 11."),
    
    8: ("apps_app_store", "high",
        "Customer says apps can no longer be updated."),
    
    9: ("other_unclear", "high",
        "Only says 'ios 11' without describing a problem."),
    
    10: ("apps_app_store", "high",
         "Phone app will not open, preventing access to calling and voicemail.")
}

for annotation_id, (intent, confidence, note) in cluster_0_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

print("Cluster 0 annotation completed.")

display(
    annotation_df[
        annotation_df["semantic_cluster"] == 0
    ][
        [
            "annotation_id",
            "customer_message",
            "human_intent",
            "annotation_confidence",
            "annotation_notes"
        ]
    ]
)

Cluster 0 annotation completed.


,annotation_id,customer_message,human_intent,annotation_confidence,annotation_notes
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,high,Delete key on Bluetooth keyboard stopped working while typing.
1,2,I am using an iPhone 8 Plus and iOS 11.1.2,other_unclear,high,Only provides device and iOS version; no problem is stated.
2,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",performance_freezing,high,Phone repeatedly freezes; camera and gallery problems are secondary.
3,4,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,software_update,high,No sound started immediately after the iOS update.
4,5,"About 2 weeks, I have IOS 11.1.2",other_unclear,high,Only states the iOS version and duration; no clear support issue.
5,6,I got a crazy security bug issue with iOS 11.1,software_update,medium,"Security bug is associated with iOS 11.1, but the exact problem is unclear."
6,7,force touch for app multitask gone in iOS11 ?,software_update,high,Customer reports a feature being unavailable in iOS 11.
7,8,Hey iOS 11 sucks none of my apps can update anymore,apps_app_store,high,Customer says apps can no longer be updated.
8,9,ios 11,other_unclear,high,Only says 'ios 11' without describing a problem.
9,10,": New iPhoneX. iOS 11.1.2 update. Phone app literally will not open. Cannot access call history, my voicemail or make a call. WTF",apps_app_store,high,"Phone app will not open, preventing access to calling and voicemail."


In [28]:
# Cell 20 — Complete missing annotation fields for Cluster 0

annotation_df.loc[
    annotation_df["annotation_id"] == 10,
    "annotation_confidence"
] = "high"

annotation_df.loc[
    annotation_df["annotation_id"] == 10,
    "annotation_notes"
] = (
    "Phone app will not open, preventing access to calling and voicemail."
)

# Save current progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 0 is fully annotated
cluster_0_check = annotation_df[
    annotation_df["semantic_cluster"] == 0
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_0_check)

print("Missing intents:",
      cluster_0_check["human_intent"].eq("").sum())

print("Missing confidence:",
      cluster_0_check["annotation_confidence"].eq("").sum())

print("Cluster 0 completed and progress saved.")

,annotation_id,human_intent,annotation_confidence,annotation_notes
0,1,keyboard_input,high,Delete key on Bluetooth keyboard stopped working while typing.
1,2,other_unclear,high,Only provides device and iOS version; no problem is stated.
2,3,performance_freezing,high,Phone repeatedly freezes; camera and gallery problems are secondary.
3,4,software_update,high,No sound started immediately after the iOS update.
4,5,other_unclear,high,Only states the iOS version and duration; no clear support issue.
5,6,software_update,medium,"Security bug is associated with iOS 11.1, but the exact problem is unclear."
6,7,software_update,high,Customer reports a feature being unavailable in iOS 11.
7,8,apps_app_store,high,Customer says apps can no longer be updated.
8,9,other_unclear,high,Only says 'ios 11' without describing a problem.
9,10,apps_app_store,high,"Phone app will not open, preventing access to calling and voicemail."


Missing intents: 0
Missing confidence: 0
Cluster 0 completed and progress saved.


In [29]:
# Cell 21 — Inspect Cluster 1 messages

cluster_1 = annotation_df[
    annotation_df["semantic_cluster"] == 1
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_1)

,annotation_id,customer_message,support_response
10,11,So is everybody shit looking like this or just mine ?,@128694 Thanks for reaching out. DM which iPhone and specific iOS version you are using to get started. https://t.co/GDrqU22YpT
11,12,Location details sent in DM !,@124483 Thank you. We will reply to you there.
12,13,"I promise that I did the workouts to finish the Thanksgiving day challenge. Yet, no badge. Does have any ideas?",@141105 We want to be sure you get the badge you earned. What country do you live in? DM us and let us know. https://t.co/GDrqU22YpT
13,14,@AppleSupport https://t.co/9KrD2oAz5L,@141848 Thanks for reaching out. We'd like some additional details to assist. Which iOS is this occurring on? Let us know and we'll take it from there.
14,15,Wow! Also in landscape! This is better than I think!,"@118246 We'd love to look into this with you! To begin, can you DM us what iOS version you're currently on? https://t.co/GDrqU22YpT"
15,16,"my watch has been saying I’ve been averaging 57 minutes, now today it’s 52, I checked and did the math it’s 57... what the???",@118099 We'd like to help. Let's take this to DM and we'll explore ways to provide you assistance. https://t.co/GDrqU22YpT
16,17,@AppleSupport https://t.co/yvu10fymce,@123687 Thank you. Please check to see if you have an update available under Settings &gt; General &gt; Software Update.
17,18,what is this?,"@136587 We can definitely offer some guidance. Since noticing this, have you had a chance to restart your iPad? This step may seem simple, but it can resolve minor issues with your device."
18,19,Thank you,@125167 You're welcome!
19,20,Thank you!,@144022 You are very welcome.


In [30]:
# Cell 22 — Add human-validated labels for Cluster 1

cluster_1_labels = {
    11: ("other_unclear", "high",
         "Customer refers to an unspecified appearance or problem without enough context."),
    
    12: ("other_unclear", "high",
         "Only says location details were sent in DM; no issue is described."),
    
    13: ("other_unclear", "high",
         "Apple Watch challenge and badge issue does not fit the current support taxonomy."),
    
    14: ("other_unclear", "high",
         "Only tags AppleSupport and provides a link; no problem is described."),
    
    15: ("other_unclear", "high",
         "Positive comment about landscape mode with no specific support problem."),
    
    16: ("other_unclear", "high",
         "Apple Watch workout-time discrepancy does not clearly fit the current taxonomy."),
    
    17: ("other_unclear", "high",
         "Customer message contains only a link and AppleSupport mention; no issue is described."),
    
    18: ("other_unclear", "high",
         "Question is too vague to identify a specific support intent."),
    
    19: ("other_unclear", "high",
         "Thank-you message with no support intent."),
    
    20: ("other_unclear", "high",
         "Thank-you message with no support intent.")
}

for annotation_id, (intent, confidence, note) in cluster_1_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

print("Cluster 1 annotation completed.")

display(
    annotation_df[
        annotation_df["semantic_cluster"] == 1
    ][
        [
            "annotation_id",
            "customer_message",
            "human_intent",
            "annotation_confidence",
            "annotation_notes"
        ]
    ]
)

Cluster 1 annotation completed.


,annotation_id,customer_message,human_intent,annotation_confidence,annotation_notes
10,11,So is everybody shit looking like this or just mine ?,other_unclear,high,Customer refers to an unspecified appearance or problem without enough context.
11,12,Location details sent in DM !,other_unclear,high,Only says location details were sent in DM; no issue is described.
12,13,"I promise that I did the workouts to finish the Thanksgiving day challenge. Yet, no badge. Does have any ideas?",other_unclear,high,Apple Watch challenge and badge issue does not fit the current support taxonomy.
13,14,@AppleSupport https://t.co/9KrD2oAz5L,other_unclear,high,Only tags AppleSupport and provides a link; no problem is described.
14,15,Wow! Also in landscape! This is better than I think!,other_unclear,high,Positive comment about landscape mode with no specific support problem.
15,16,"my watch has been saying I’ve been averaging 57 minutes, now today it’s 52, I checked and did the math it’s 57... what the???",other_unclear,high,Apple Watch workout-time discrepancy does not clearly fit the current taxonomy.
16,17,@AppleSupport https://t.co/yvu10fymce,other_unclear,high,Customer message contains only a link and AppleSupport mention; no issue is described.
17,18,what is this?,other_unclear,high,Question is too vague to identify a specific support intent.
18,19,Thank you,other_unclear,high,Thank-you message with no support intent.
19,20,Thank you!,other_unclear,high,Thank-you message with no support intent.


In [31]:
# Cell 23 — Inspect Cluster 2 messages

cluster_2 = annotation_df[
    annotation_df["semantic_cluster"] == 2
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_2)

,annotation_id,customer_message,support_response
20,21,11.0.2,@129985 We’re here to help. Let us know via DM if this happens after going to Settings &gt; Keyboards &gt; Predictive and turn it off. https://t.co/GDrqU22YpT
21,22,11.1.1,"@141849 Thanks for that detail. Let's get you up to speed as a first step and see if the issues resolve themselves. We've released iOS 11.1.2 recently, so let's get you to back up and update first. Check out this link for assistance in this step: https://t.co/EGo8nZttAk"
22,23,11.1.2,@140197 Thanks for that. Would you mind joining us in DM please? We'd like to continue our conversation there. https://t.co/GDrqU22YpT
23,24,"Yes it did, 11.0.3","@115868 We've just released iOS 11.1. We suggest backing up &amp; updating. Then, test the issue again. Here's how: https://t.co/80YRnjDFDk"
24,25,11.1,@128332 Are you using a third-party keyboard in Settings &gt; General &gt; Keyboard &gt; Keyboards ? Let us know in DM. https://t.co/GDrqU22YpT
25,26,11.1,"@129051 Great, thanks! Have you tried to restart your iPhone or check to ensure your apps are up-to-date? If not, try that for us."
26,27,Hey this is what happened after 10.13.1 please advise.!!,@123175 We want your Mac to work as expected. Does this message come back up if you click the Restart button?
27,28,it is updated to 11.1,@124862 Thank you. Please send us a DM letting us know when this first started happening and we'll take it from there. https://t.co/GDrqU22YpT
28,29,I have 11.0.3,"@117650 The first step we want to take is to back it up, then update it to iOS 11.1 to rule that out: https://t.co/80YRnjDFDk"
29,30,It started in may. 11.1.2,"@142515 Check out the following article and try out the steps it suggestsL: https://t.co/5KootZbxj1\n\nIf this does not resolve the issue, let us know via DM which country you are in and who your carrier is, then we can go from there. https://t.co/GDrqU22YpT"


In [32]:
# Cell 24 — Add human-validated labels for Cluster 2

cluster_2_labels = {
    21: ("other_unclear", "high",
         "Only provides iOS version 11.0.2; no problem is described."),
    
    22: ("other_unclear", "high",
         "Only provides iOS version 11.1.1; no specific issue is stated."),
    
    23: ("other_unclear", "high",
         "Only provides iOS version 11.1.2."),
    
    24: ("other_unclear", "medium",
         "Reply refers to missing context; the version alone does not identify the intent."),
    
    25: ("other_unclear", "high",
         "Only provides iOS version 11.1; no problem is described."),
    
    26: ("other_unclear", "high",
         "Only provides iOS version 11.1; no problem is described."),
    
    27: ("software_update", "medium",
         "Customer explicitly connects the problem to macOS 10.13.1, but the actual problem is not stated."),
    
    28: ("other_unclear", "high",
         "Only says the device is updated to iOS 11.1; no problem is described."),
    
    29: ("other_unclear", "high",
         "Only provides iOS version 11.0.3; no problem is described."),
    
    30: ("other_unclear", "medium",
         "Provides when something started and the iOS version, but the actual issue is missing.")
}

for annotation_id, (intent, confidence, note) in cluster_2_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

print("Cluster 2 annotation completed.")

display(
    annotation_df[
        annotation_df["semantic_cluster"] == 2
    ][
        [
            "annotation_id",
            "customer_message",
            "human_intent",
            "annotation_confidence",
            "annotation_notes"
        ]
    ]
)

Cluster 2 annotation completed.


,annotation_id,customer_message,human_intent,annotation_confidence,annotation_notes
20,21,11.0.2,other_unclear,high,Only provides iOS version 11.0.2; no problem is described.
21,22,11.1.1,other_unclear,high,Only provides iOS version 11.1.1; no specific issue is stated.
22,23,11.1.2,other_unclear,high,Only provides iOS version 11.1.2.
23,24,"Yes it did, 11.0.3",other_unclear,medium,Reply refers to missing context; the version alone does not identify the intent.
24,25,11.1,other_unclear,high,Only provides iOS version 11.1; no problem is described.
25,26,11.1,other_unclear,high,Only provides iOS version 11.1; no problem is described.
26,27,Hey this is what happened after 10.13.1 please advise.!!,software_update,medium,"Customer explicitly connects the problem to macOS 10.13.1, but the actual problem is not stated."
27,28,it is updated to 11.1,other_unclear,high,Only says the device is updated to iOS 11.1; no problem is described.
28,29,I have 11.0.3,other_unclear,high,Only provides iOS version 11.0.3; no problem is described.
29,30,It started in may. 11.1.2,other_unclear,medium,"Provides when something started and the iOS version, but the actual issue is missing."


In [33]:
# Cell 25 — Inspect Cluster 3 messages

cluster_3 = annotation_df[
    annotation_df["semantic_cluster"] == 3
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_3)

,annotation_id,customer_message,support_response
30,31,@115858 creates amazing technology..... but talk to text still messes up 90% of what I will it to type..... #fixyourcrap,@117101 Let’s work together on this. How long has this been happening? Do you experience similar behavior with Siri?
31,32,Anybody else experiencing the bug where the letter I turning into a weird set of glyphs? Igate,@127936 That's not the experience we want you have with iOS 11. Follow us to DM and we'll get started. https://t.co/GDrqU22YpT
32,33,Man I’m having the same problem. Thought EYE was alone. Was just about to accept life without ever using the word “I” anymore.,"@118365 We want to take a look into this with you. DM us your current iOS version, and we’ll start there. https://t.co/GDrqU22YpT"
33,34,Okay I thought it was just my phone fix this shit with the letter EYE,@123990 Thanks for reaching out. DM which iPhone and specific iOS version you are using to get started. https://t.co/GDrqU22YpT
34,35,hey @115858 get ur shit together so i can type “i” without this stupid problem https://t.co/43iHbkJK6K,@130490 We're here for you. Please DM us the version of iOS you're using and we can go from there together. https://t.co/GDrqU22YpT
35,36,Why does my iPhone randomly capitalise letters? iOS,"@136074 We'd love the chance to look into this with you. Which iOS version are you using on your iPhone? Also, could you give us an example of when you're seeing letters being capitalized?"
36,37,For those that have been wondering. why are y’all punishing us for using pronouns!?,@123676 Thanks for reaching out. DM which iPhone and specific iOS version you are using to get started. https://t.co/GDrqU22YpT
37,38,whenever I type the letter “i” it auto corrects to a “a ?” What is going on!????,@123062 Let us help you get this sorted out. Join us in DM so we can better assist you: https://t.co/GDrqU22YpT
38,39,"The only keyboard I have is whatever my iPhone came with. It’s occurring in everything: messages, social media, emailing ect.",@118366 We'd like to take a closer look. Send us a DM and we can start there. https://t.co/GDrqU22YpT
39,40,"i.t really sucks that every time i go to type the word i.t, i.t autocorrects so there’s a stupid ass period in the middle of i.t. @115858 what is going on",@134162 We'd be happy to look into this with you more in detail. Please reach out to us via DM and we'll get started. https://t.co/GDrqU22YpT


In [34]:
# Cell 26 — Add human-validated labels for Cluster 3

cluster_3_labels = {
    31: ("keyboard_input", "high",
         "Talk-to-text produces incorrect typed text, making this a text-input problem."),
    
    32: ("keyboard_input", "high",
         "The letter 'I' is converted into incorrect glyphs while typing."),
    
    33: ("keyboard_input", "high",
         "Customer reports the same letter 'I' typing problem."),
    
    34: ("keyboard_input", "high",
         "Customer cannot type the letter 'I' correctly."),
    
    35: ("keyboard_input", "high",
         "Typing the letter 'i' causes incorrect autocorrect behavior."),
    
    36: ("keyboard_input", "high",
         "iPhone randomly capitalizes letters while the customer is typing."),
    
    37: ("keyboard_input", "medium",
         "Message refers to a typing/pronoun issue, but the exact technical problem is somewhat implicit."),
    
    38: ("keyboard_input", "high",
         "The letter 'i' is automatically changed while typing across multiple apps."),
    
    39: ("keyboard_input", "high",
         "Autocorrect inserts a period while the customer is typing.")
}

for annotation_id, (intent, confidence, note) in cluster_3_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

print("Cluster 3 annotation completed.")

display(
    annotation_df[
        annotation_df["semantic_cluster"] == 3
    ][
        [
            "annotation_id",
            "customer_message",
            "human_intent",
            "annotation_confidence",
            "annotation_notes"
        ]
    ]
)

Cluster 3 annotation completed.


,annotation_id,customer_message,human_intent,annotation_confidence,annotation_notes
30,31,@115858 creates amazing technology..... but talk to text still messes up 90% of what I will it to type..... #fixyourcrap,keyboard_input,high,"Talk-to-text produces incorrect typed text, making this a text-input problem."
31,32,Anybody else experiencing the bug where the letter I turning into a weird set of glyphs? Igate,keyboard_input,high,The letter 'I' is converted into incorrect glyphs while typing.
32,33,Man I’m having the same problem. Thought EYE was alone. Was just about to accept life without ever using the word “I” anymore.,keyboard_input,high,Customer reports the same letter 'I' typing problem.
33,34,Okay I thought it was just my phone fix this shit with the letter EYE,keyboard_input,high,Customer cannot type the letter 'I' correctly.
34,35,hey @115858 get ur shit together so i can type “i” without this stupid problem https://t.co/43iHbkJK6K,keyboard_input,high,Typing the letter 'i' causes incorrect autocorrect behavior.
35,36,Why does my iPhone randomly capitalise letters? iOS,keyboard_input,high,iPhone randomly capitalizes letters while the customer is typing.
36,37,For those that have been wondering. why are y’all punishing us for using pronouns!?,keyboard_input,medium,"Message refers to a typing/pronoun issue, but the exact technical problem is somewhat implicit."
37,38,whenever I type the letter “i” it auto corrects to a “a ?” What is going on!????,keyboard_input,high,The letter 'i' is automatically changed while typing across multiple apps.
38,39,"The only keyboard I have is whatever my iPhone came with. It’s occurring in everything: messages, social media, emailing ect.",keyboard_input,high,Autocorrect inserts a period while the customer is typing.
39,40,"i.t really sucks that every time i go to type the word i.t, i.t autocorrects so there’s a stupid ass period in the middle of i.t. @115858 what is going on",,,


In [35]:
# Cell 27 — Complete Cluster 3 annotation

annotation_df.loc[
    annotation_df["annotation_id"] == 40,
    "human_intent"
] = "keyboard_input"

annotation_df.loc[
    annotation_df["annotation_id"] == 40,
    "annotation_confidence"
] = "high"

annotation_df.loc[
    annotation_df["annotation_id"] == 40,
    "annotation_notes"
] = (
    "Autocorrect changes text while typing, specifically inserting "
    "a period into the word 'i.t'."
)

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 3 is completely annotated
cluster_3_check = annotation_df[
    annotation_df["semantic_cluster"] == 3
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence"
    ]
]

display(cluster_3_check)

print(
    "Missing labels in Cluster 3:",
    cluster_3_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence
30,31,keyboard_input,high
31,32,keyboard_input,high
32,33,keyboard_input,high
33,34,keyboard_input,high
34,35,keyboard_input,high
35,36,keyboard_input,high
36,37,keyboard_input,medium
37,38,keyboard_input,high
38,39,keyboard_input,high
39,40,keyboard_input,high


Missing labels in Cluster 3: 0


In [36]:
# Cell 28 — Inspect Cluster 4 messages

cluster_4 = annotation_df[
    annotation_df["semantic_cluster"] == 4
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_4)

,annotation_id,customer_message,support_response
40,41,Thanks! But it used to be so nice when I could limit the number of episodes downloaded but still have some pre-downloaded for flights n things. Is the setting coming back?,"@121033 While we don't have a direct answer for you today, we’d encourage you to leave feedback at:\nhttps://t.co/9WSVjlVyvt"
41,42,thanks. I read the document & as it is a 3-series watch I’ve checked the settings as instructed - they were set correctly Do I need to go through the recalibration process as per a 1-series watch? Why doesn’t Health app just point to same XML field as Activity app?,@136062 Thank you for those details. Have you restarted both your iPhone and your Apple Watch since noticing this? DM us and let us know. https://t.co/GDrqU22YpT
42,43,I have not installed any new appliaction. Anyway automatic application updates are enabled so I cannot determine which of the ones that were installed today might be coincident with the exact time frame when the issue appeared.,@142511 Got it. Let's look into this further via DM. Let us know if you're able to access the Control Center preferences via Settings. https://t.co/GDrqU22YpT
43,44,I need your help! I want to play animal crossing but the app immediately crashes when I open it,"@128371 We want to make sure you are able to play the games you'd like. In order to assist you, we need some information from you. What kind of device are you using? What version of the software is installed? Meet us with this information, here: https://t.co/GDrqU22YpT"
44,45,"It’s become too slow ,apps not working properly",@125183 Let's get you back to enjoying your device. What kind of Apple device are you using? https://t.co/GDrqU22YpT
45,46,Does anyone EVER get Apple’s ‘reading list’ to work offline in Safari? What a load of old . Any alternatives that work?,@124688 We'd love to look into this with you. Tell us what happens while trying to access your Reading List while offline.
46,47,i have 256 hd and timemachine/system is taking up 150gb of space. I only have 25gb free. Is it suppose to work like this?,@117459 Thanks for reaching out to us. We're happy to help out. We recommend checking out this link: https://t.co/03O2CionXW
47,48,My husband hasn’t updated his MacBook since like 2011... do you think I could update it and be safe?,"@123974 As long as his Mac is compatible, it shouldn't be a problem. Be sure he has a backup. https://t.co/109nnHwfMC"
48,49,When i go to upload to social media with a slo-mo or screen shotted picture i am getting a loading circle in the middle of the screen and it never finishes loading. Doesnt allow upload to complete,"@139254 Thank you. Does the issue persist if you restart your device? Also, which software version are you using? You can tap Settings &gt; General &gt; About to locate the software version."
49,50,all of a sudden I am getting emails pushed via apple mail- I’ve had them set as fetch for a long time. Can’t find where to change back to fetch. The only change I have made is software update. HALP! I don’t want all these to push.,"@122014 We can understand not wanting the incoming emails to push through. We'll get to the root cause of this, and find you the information you need.\n\nIn order to do so, we have some questions for you. Please meet us in DM with your current iOS and we'll go from there. https://t.co/GDrqU22YpT"


In [37]:
# Cell 29 — Add human-validated labels for Cluster 4

cluster_4_labels = {
    41: ("apps_app_store", "high",
         "Customer asks about an app-related setting for controlling downloaded episodes."),
    
    42: ("other_unclear", "high",
         "Apple Watch and Health XML-field issue does not clearly fit the current taxonomy."),
    
    43: ("other_unclear", "high",
         "Customer provides context about automatic app updates, but the actual support problem is not clearly stated."),
    
    44: ("apps_app_store", "high",
         "Animal Crossing crashes immediately when the customer opens the app."),
    
    45: ("performance_freezing", "high",
         "Main complaint is that the device has become slow; app problems are secondary."),
    
    46: ("apps_app_store", "high",
         "Customer is asking about Safari Reading List functionality while offline."),
    
    47: ("other_unclear", "high",
         "Time Machine and system storage usage does not clearly fit the current support taxonomy."),
    
    48: ("apps_app_store", "high",
         "Customer cannot complete an upload through a social-media app."),
    
    49: ("software_update", "high",
         "Customer explicitly connects the problem to a software update."),
    
    50: ("software_update", "high",
         "Customer says the Apple Mail behavior changed after a software update.")
}

for annotation_id, (intent, confidence, note) in cluster_4_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 4
cluster_4_check = annotation_df[
    annotation_df["semantic_cluster"] == 4
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_4_check)

print(
    "Missing labels in Cluster 4:",
    cluster_4_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
40,41,apps_app_store,high,Customer asks about an app-related setting for controlling downloaded episodes.
41,42,other_unclear,high,Apple Watch and Health XML-field issue does not clearly fit the current taxonomy.
42,43,other_unclear,high,"Customer provides context about automatic app updates, but the actual support problem is not clearly stated."
43,44,apps_app_store,high,Animal Crossing crashes immediately when the customer opens the app.
44,45,performance_freezing,high,Main complaint is that the device has become slow; app problems are secondary.
45,46,apps_app_store,high,Customer is asking about Safari Reading List functionality while offline.
46,47,other_unclear,high,Time Machine and system storage usage does not clearly fit the current support taxonomy.
47,48,apps_app_store,high,Customer cannot complete an upload through a social-media app.
48,49,software_update,high,Customer explicitly connects the problem to a software update.
49,50,software_update,high,Customer says the Apple Mail behavior changed after a software update.


Missing labels in Cluster 4: 0


In [38]:
# Cell 30 — Inspect Cluster 5 messages

cluster_5 = annotation_df[
    annotation_df["semantic_cluster"] == 5
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_5)

,annotation_id,customer_message,support_response
50,51,"Depende como responda esto perderán un cliente (+familia) y prescriptor desde hace más de 15 años, usuario de toda la gama",@124301 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs
51,52,"leg dit even uit? Dit is mijn oud e-mail adress/apple id, sinds een recente update krijg ik constant deze melding. Ik heb sinds paar jaar een nieuw e-mail adres en ook een apple id op dit adres. Ik was (nog altijd) het wachtwoord kwijt van het oude e-mail & apple-id",@138153 We offer support via Twitter in English. Contact us for help in your preferred language here: https://t.co/IBIY3vMgPj
52,53,cuando vais arreglar el problema de la batería con vuestras actualizaciones de IOS? Es de vergüenza,@124305 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs
53,54,Soy sólo yo...o a ustedes también les pasa que a toda hora se les enciende el WIFI del teléfono desde la nueva actualización del iOS ?,@142165 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs
54,55,"Ongelofelijk dat een MacBook Pro al na 2,5 jaar kapot gaat. Als antwoord krijg je dan alle kosten inclusief reparatie zijn voor eigen kosten. Terwijl de winkel al zegt u kan er niks aan doen dat hij stuk is. slechte service apple geenservice macbook pro",@140684 We offer support via Twitter in English. Contact an Advisor for help in your preferred language here: https://t.co/IBIY3vMgPj
55,56,E eu não recebo notificação de nenhum aplicativo,"@138173 Since our Twitter support is available in English, get help at https://t.co/IBIY3vMgPj or join https://t.co/pvaOFfPbjt"
56,57,"Klucííí... z @115858, co mi to děláte? Safari s #iOS 11.1 na #iPhone8Plus neustále padá 📲🕳 chápu, je podzim 🍏🍂...ale tohle nebaví! 🤨 #Fail",@123840 We offer support via Twitter in English. Contact us for help in your preferred language here: https://t.co/IBIY3vMgPj
57,58,@115858 Estoy muy decepcionado con la calidad de la compra online. Despues de esperar tanto tiempo para el IPhone X + AppleWatch teneis problemas con el sistema de pago. Muy muy decepcionado !!! Espero que esto no afecte a la fecha de entrega de mi pedido!!!,@138716 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs
58,59,Desde que actualice a iOS 11 mi iPhone se volvió lento. Me recuerda a los viejos Samsung! Que está pasando?,@127761 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs
59,60,"Bienvenida sea la multi tarea al 3D touch en iphone, gracias newupdate",@117646 We offer support via Twitter in English. Get help in Spanish here: https://t.co/IBIY3vMgPj or join https://t.co/OczyRx7IOs


In [39]:
# Cell 31 — Add human-validated labels for Cluster 5

cluster_5_labels = {
    51: ("other_unclear", "high",
         "Customer expresses dissatisfaction but does not state a specific technical support problem."),
    
    52: ("apple_id_icloud", "high",
         "Customer discusses an old Apple ID, old email address, and a forgotten password."),
    
    53: ("battery_power", "high",
         "Customer explicitly complains about battery problems caused by iOS updates."),
    
    54: ("connectivity", "high",
         "Customer reports that Wi-Fi keeps turning on after a new iOS update."),
    
    55: ("device_hardware", "high",
         "Customer reports a MacBook Pro hardware failure and discusses repair costs."),
    
    56: ("other_unclear", "high",
         "Customer reports missing app notifications, which does not clearly fit the current taxonomy."),
    
    57: ("performance_freezing", "high",
         "Customer reports Safari repeatedly crashing on iOS 11.1."),
    
    58: ("other_unclear", "high",
         "Customer is discussing an online purchase, payment system, and delivery timing, which are outside the current taxonomy."),
    
    59: ("performance_freezing", "high",
         "Customer reports that the iPhone became slow after updating to iOS 11."),
    
    60: ("other_unclear", "medium",
         "Customer mentions multitasking and 3D Touch positively but does not describe a specific support problem.")
}

for annotation_id, (intent, confidence, note) in cluster_5_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 5
cluster_5_check = annotation_df[
    annotation_df["semantic_cluster"] == 5
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_5_check)

print(
    "Missing labels in Cluster 5:",
    cluster_5_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
50,51,other_unclear,high,Customer expresses dissatisfaction but does not state a specific technical support problem.
51,52,apple_id_icloud,high,"Customer discusses an old Apple ID, old email address, and a forgotten password."
52,53,battery_power,high,Customer explicitly complains about battery problems caused by iOS updates.
53,54,connectivity,high,Customer reports that Wi-Fi keeps turning on after a new iOS update.
54,55,device_hardware,high,Customer reports a MacBook Pro hardware failure and discusses repair costs.
55,56,other_unclear,high,"Customer reports missing app notifications, which does not clearly fit the current taxonomy."
56,57,performance_freezing,high,Customer reports Safari repeatedly crashing on iOS 11.1.
57,58,other_unclear,high,"Customer is discussing an online purchase, payment system, and delivery timing, which are outside the current taxonomy."
58,59,performance_freezing,high,Customer reports that the iPhone became slow after updating to iOS 11.
59,60,other_unclear,medium,Customer mentions multitasking and 3D Touch positively but does not describe a specific support problem.


Missing labels in Cluster 5: 0


In [40]:
# Cell 32 — Inspect Cluster 6 messages

cluster_6 = annotation_df[
    annotation_df["semantic_cluster"] == 6
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_6)

,annotation_id,customer_message,support_response
60,61,how can i reset security questions? i face this error: We don't have sufficient information to reset your security questions.,@124485 We'll need to refer you here for help with your Apple ID: https://t.co/UJm15vmMM2
61,62,Yeah I actually just fixed it by tapping the top of the phone above the camera. Works fine again! Thanks!,@140212 Excellent! You're welcome. Have a great day!
62,63,I’ve tried trashing Bluetooth prefs - but this didn’t help. Could you please explain how to run the hardware test?,"@139271 Absolutely, there are a few steps involved. Are you able to open the article we previously sent? If so, the steps are there. If you have any issue viewing the article send us a DM with the link below and we'll provide them for you. https://t.co/GDrqU22YpT"
63,64,Update: still resets phone even after removing and reinstalling.,"@136097 We appreciate the update. Just to be sure that we're on the same page, is your iPhone restarting while you're using any other apps? Please send us a DM with the results. https://t.co/GDrqU22YpT"
64,65,How do I get rid of this notification?,@118244 Let's see how we can help. Is that the message you see when trying to swipe the notification and get to the app?
65,66,Once I removed two step authentication it all went back to normal - thanks for responding,@125180 We're here to assist if you need assistance in the future.
66,67,My laptop’s doing this again,@143056 We want to help. Let's continue in DM. https://t.co/GDrqU22YpT
67,68,That worked. Thank you.,@122925 You're very welcome. Please let us know if there's anything else we can do for you.
68,69,Its slowing down my mac a lot,"@117459 Is 25 GBs what you have left after the snapshots were created on the drive though? Or, were you low on space prior to snapshots?"
69,70,"Update to latest version,did the restart thing but still no vibrate at all when notifications coming",@122940 Let's look into this some more. Send us a DM and let us know anything else you have tried as well as the country you're in. https://t.co/GDrqU22YpT


In [42]:
# Cell 33 — Add human-validated labels for Cluster 6

cluster_6_labels = {
    61: ("apple_id_icloud", "high",
         "Security questions are part of Apple ID account recovery and security."),
    
    62: ("device_hardware", "medium",
         "Customer asks about a hardware test and reports that physically tapping the phone restored normal behavior."),
    
    63: ("connectivity", "high",
         "Customer is troubleshooting a Bluetooth-related problem and has already tried removing Bluetooth preferences."),
    
    64: ("performance_freezing", "high",
         "Customer reports that the phone continues to restart/reset even after reinstalling."),
    
    65: ("other_unclear", "high",
         "Customer asks about removing a notification but does not provide enough detail to map it to the current taxonomy."),
    
    66: ("apple_id_icloud", "high",
         "Customer reports that removing two-step authentication resolved the problem, indicating an account security issue."),
    
    67: ("other_unclear", "high",
         "Customer says the laptop is doing something again but does not describe the actual problem."),
    
    68: ("other_unclear", "high",
         "Customer only confirms that a previous solution worked and does not state the original problem."),
    
    69: ("performance_freezing", "high",
         "Customer reports that the Mac has become significantly slower."),
    
    70: ("other_unclear", "medium",
         "Customer reports missing notification vibration, which is outside the current taxonomy; the update is context rather than the primary issue.")
}

for annotation_id, (intent, confidence, note) in cluster_6_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 6
cluster_6_check = annotation_df[
    annotation_df["semantic_cluster"] == 6
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_6_check)

print(
    "Missing labels in Cluster 6:",
    cluster_6_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
60,61,apple_id_icloud,high,Security questions are part of Apple ID account recovery and security.
61,62,device_hardware,medium,Customer asks about a hardware test and reports that physically tapping the phone restored normal behavior.
62,63,connectivity,high,Customer is troubleshooting a Bluetooth-related problem and has already tried removing Bluetooth preferences.
63,64,performance_freezing,high,Customer reports that the phone continues to restart/reset even after reinstalling.
64,65,other_unclear,high,Customer asks about removing a notification but does not provide enough detail to map it to the current taxonomy.
65,66,apple_id_icloud,high,"Customer reports that removing two-step authentication resolved the problem, indicating an account security issue."
66,67,other_unclear,high,Customer says the laptop is doing something again but does not describe the actual problem.
67,68,other_unclear,high,Customer only confirms that a previous solution worked and does not state the original problem.
68,69,performance_freezing,high,Customer reports that the Mac has become significantly slower.
69,70,other_unclear,medium,"Customer reports missing notification vibration, which is outside the current taxonomy; the update is context rather than the primary issue."


Missing labels in Cluster 6: 0


In [43]:
# Cell 34 — Inspect Cluster 7 messages

cluster_7 = annotation_df[
    annotation_df["semantic_cluster"] == 7
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_7)

,annotation_id,customer_message,support_response
70,71,"Battery life awful, iOS 11.2, iPhone X","@141455 We’d love to help with your battery life. To start, please send us a DM. https://t.co/GDrqU22YpT"
71,72,"I have a 6 Plus, everyone that I have talked to has been having the issues that he mentioned since the iOS 11 updates. My battery life is terrible now, my cursor/screen freezes & jumps around while I’m typing & I have to do a hard reset every couple of days.","@144218 We expect you to have a great experience running the latest version of iOS, 11.1.2, on your iPhone 6 Plus. Send us a Direct Message, and we'll work together to investigate the behaviors you're running into, so we can offer the best solution. https://t.co/GDrqU22YpT"
72,73,"So I accidentally did the new @115858 update,cause it kept popping up on my screen,n now battery is at 39% after charging all night n went down to 30% in literally 15seconds of use..... WTF?!",@135638 We certainly understand how important a good battery life is. We'd be happy to look into this further with you. \nWhich device are we working with? Shoot us a DM and let us know. https://t.co/GDrqU22YpT
73,74,Even i have the battery drain issues since updating to iOS 11. Not even any better on iOS 11.1,"@124883 Battery life is important, so we'd like to help. Let us know in DM when this started happening. https://t.co/GDrqU22YpT"
74,75,you new update is crap it just drains the battery. 100%full 5 mins late 88%,"@135084 That's certainly not how we want the battery to be working for you. Let's work together. To start, DM us the device you're using and how many times you charge the device on a daily basis. https://t.co/GDrqU22YpT"
75,76,"Hey 11.2.1 is killing my battery- big time. iPhone 6s+ I used to make it to 3 pm and still have 80%. I’m below 40 already (3:15). And no, I don’t use the app.",@141087 Thanks for contacting us. We know how important battery life is. We'd be happy to help in any way we can. Let's meet in DM to troubleshoot together. https://t.co/GDrqU22YpT
76,77,"My 6s starts as expected at 100% use it for 5 minutes and it needs recharged, really infuriating",@142823 We'd like to look into what's happening on your iPhone. Could you join us in DM to get started? https://t.co/GDrqU22YpT
77,78,"my iphone 6s is rendered two 2 hours. Drained battery, killed all non used apps and still two hours. newiosblows",@123705 Let's look into what's happening together. Send us a DM telling us the country you're located in to begin. https://t.co/GDrqU22YpT
78,79,Please help. My mom's iPhone 6s Plus' battery drains so fast even without using it. It happened after she updated to iOS 11.1.2.,@130909 Thanks for reaching out to us. We are always happy to help. Send us a DM so we can look into this issue with you there. https://t.co/GDrqU22YpT
79,80,Why are all my Apple gadgets slowing down all of the sudden,@135578 We want to help. What do you mean by gadgets?


In [44]:
# Cell 35 — Add human-validated labels for Cluster 7

cluster_7_labels = {
    71: ("battery_power", "high",
         "Customer explicitly reports very poor battery life on iOS 11.2."),
    
    72: ("battery_power", "high",
         "Main complaint is severe battery drain after the iOS 11 update; freezing and typing issues are secondary."),
    
    73: ("battery_power", "high",
         "Battery loses charge extremely quickly after the customer installed a new update."),
    
    74: ("battery_power", "high",
         "Customer explicitly reports battery drain after updating to iOS 11."),
    
    75: ("battery_power", "high",
         "Battery drops from 100% to 88% within minutes, indicating severe battery drain."),
    
    76: ("battery_power", "high",
         "Customer reports unusually rapid battery drain on an iPhone 6s Plus."),
    
    77: ("battery_power", "high",
         "Customer reports needing to recharge the phone after only a few minutes of use."),
    
    78: ("battery_power", "high",
         "Customer reports severe battery drain even when the phone is not being used."),
    
    79: ("battery_power", "high",
         "Customer reports that the iPhone battery drains extremely quickly."),
    
    80: ("performance_freezing", "medium",
         "Customer reports that Apple devices have suddenly become slower, with no specific battery complaint.")
}

for annotation_id, (intent, confidence, note) in cluster_7_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 7
cluster_7_check = annotation_df[
    annotation_df["semantic_cluster"] == 7
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_7_check)

print(
    "Missing labels in Cluster 7:",
    cluster_7_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
70,71,battery_power,high,Customer explicitly reports very poor battery life on iOS 11.2.
71,72,battery_power,high,Main complaint is severe battery drain after the iOS 11 update; freezing and typing issues are secondary.
72,73,battery_power,high,Battery loses charge extremely quickly after the customer installed a new update.
73,74,battery_power,high,Customer explicitly reports battery drain after updating to iOS 11.
74,75,battery_power,high,"Battery drops from 100% to 88% within minutes, indicating severe battery drain."
75,76,battery_power,high,Customer reports unusually rapid battery drain on an iPhone 6s Plus.
76,77,battery_power,high,Customer reports needing to recharge the phone after only a few minutes of use.
77,78,battery_power,high,Customer reports severe battery drain even when the phone is not being used.
78,79,battery_power,high,Customer reports that the iPhone battery drains extremely quickly.
79,80,performance_freezing,medium,"Customer reports that Apple devices have suddenly become slower, with no specific battery complaint."


Missing labels in Cluster 7: 0


In [45]:
# Cell 36 — Inspect Cluster 8 messages

cluster_8 = annotation_df[
    annotation_df["semantic_cluster"] == 8
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_8)

,annotation_id,customer_message,support_response
80,81,Always wait. I stupidly didn't. Same problem. @115858 https://t.co/HyqLZTzDbr,@136059 We're here to help. Click here to meet us in DM and we'll do everything that we can to get this sorted with your battery life. https://t.co/GDrqU22YpT
81,82,@126993 @115858 Even this tweet has the error 🙄. Fix it https://t.co/VRGWHCCsQF,@126992 We'd be happy to look into this with you. Send us a DM and we can start there. https://t.co/GDrqU22YpT
82,83,@116346 @115858 @115714 Yes a legit nothing fixes it for me,@128775 We'll be happy to look into this issue for you concerning you device. Please shoot us a DM. -AS
83,84,"It’s been a few days since @115858 made me update my phone’s operating system. Now constantly glitching, hmm 🤔 I’m shocked!",@116336 Hi there! iOS 11.1 was released earlier today containing some bug fixes. Let's back up and update: https://t.co/80YRnjDFDk
84,85,Sorry Twitter won't let me DM you because I refuse to give my phone number to a company that cannot safe guard data. I'm on 11.1.1 and refuse to upgrade to 11.1.2 because of the huge security flaw.,"@128527 We'd like to help you out. If you can't DM, there are other support options here:\n\n https://t.co/IBIY3vMgPj https://t.co/GDrqU22YpT"
85,86,my phone’s updated tho 🤔 what kinda bug is this @115858 fix it https://t.co/g3DalWEd91,@139265 We're here to help find the best solution. Can you send us a DM with detials on what you're referring to? https://t.co/GDrqU22YpT
86,87,@143278 The fingerprint has never worked for me either! C’mon @115858 Fix it!...,"@143277 Hey, let's meet up to discuss any issues you're having with your Apple products today. Here's a link to discuss this in DM: https://t.co/GDrqU22YpT"
87,88,@115858 this “I.T” thing gonna be fixed any time soon....,@136611 We can help. Let's work on this together. Send us a DM. https://t.co/GDrqU22YpT
88,89,@115858 ever since the update I’ve been experience some difficulties #Typical #MylastIphone https://t.co/BeCpD21A2D,"@136633 We appreciate you reaching back out. Are you able to adjust your keyboard using the steps in the ""Turn on one-handed typing” section of this article? https://t.co/tWyRHFS2AX Also, approximately how often do you experience that behavior with the lock screen?"
89,90,"@115858 trying to cancel an online order, each time I login i'm told I don't have access to the order.",@122933 Hello. Our Sales Support team will be happy to assist you with this issue here: https://t.co/RmFSlCBcIE


In [46]:
# Cell 37 — Add human-validated labels for Cluster 8

cluster_8_labels = {
    81: ("battery_power", "medium",
         "Customer says they have the same problem, and the support response identifies the issue as battery life, but the customer message itself provides limited detail."),
    
    82: ("other_unclear", "high",
         "Customer reports an error but does not explain what the error is."),
    
    83: ("other_unclear", "high",
         "Customer says nothing fixes the problem but does not identify the underlying issue."),
    
    84: ("software_update", "high",
         "Customer is discussing iOS versions, updating, and a security problem with the newer update."),
    
    85: ("other_unclear", "high",
         "Customer mentions a bug but does not provide enough information to identify the affected feature or problem."),
    
    86: ("device_hardware", "high",
         "Customer reports that fingerprint/Touch ID has never worked, indicating a device-level biometric issue."),
    
    87: ("keyboard_input", "high",
         "Customer explicitly refers to the 'I.T' typing/autocorrect problem."),
    
    88: ("keyboard_input", "high",
         "The issue is related to keyboard behavior and typing; the support response also directs the customer to keyboard settings."),
    
    89: ("other_unclear", "medium",
         "Customer says they have experienced difficulties since an update but does not clearly state what the problem is."),
    
    90: ("other_unclear", "high",
         "Customer is asking about cancelling an online order and accessing order information, which is outside the current technical-support taxonomy.")
}

for annotation_id, (intent, confidence, note) in cluster_8_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id
    
    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 8
cluster_8_check = annotation_df[
    annotation_df["semantic_cluster"] == 8
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_8_check)

print(
    "Missing labels in Cluster 8:",
    cluster_8_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
80,81,battery_power,medium,"Customer says they have the same problem, and the support response identifies the issue as battery life, but the customer message itself provides limited detail."
81,82,other_unclear,high,Customer reports an error but does not explain what the error is.
82,83,other_unclear,high,Customer says nothing fixes the problem but does not identify the underlying issue.
83,84,software_update,high,"Customer is discussing iOS versions, updating, and a security problem with the newer update."
84,85,other_unclear,high,Customer mentions a bug but does not provide enough information to identify the affected feature or problem.
85,86,device_hardware,high,"Customer reports that fingerprint/Touch ID has never worked, indicating a device-level biometric issue."
86,87,keyboard_input,high,Customer explicitly refers to the 'I.T' typing/autocorrect problem.
87,88,keyboard_input,high,The issue is related to keyboard behavior and typing; the support response also directs the customer to keyboard settings.
88,89,other_unclear,medium,Customer says they have experienced difficulties since an update but does not clearly state what the problem is.
89,90,other_unclear,high,"Customer is asking about cancelling an online order and accessing order information, which is outside the current technical-support taxonomy."


Missing labels in Cluster 8: 0


In [47]:
# Cell 38 — Inspect Cluster 9 messages

cluster_9 = annotation_df[
    annotation_df["semantic_cluster"] == 9
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_9)

,annotation_id,customer_message,support_response
90,91,"Hi there, my iPhone 6S plus can’t do backup for it, not by iCloud and iTunes , any way to fix it ,thanks",@123977 We'd love to help. What happens when you try back up via iTunes?
91,92,"how come @115858 that is suppose to be a top tech company, cant manage to create an update that doesn't make my iphone slower, and freezing constantly after updating it? my iphone version is only one from the last, and i have enough memory left... $ not worth",@122015 We can help with iPhone performance. What device are you currently using? We'll help get that resolved.
92,93,My worst fear came true... my iPhone alarm isn’t working and never went off this morning,"@121044 Hello. We are here to help. Has this happened previously. Also, what is your iOS version?"
93,94,Did the latest update. Problem continues when I FaceTime my wife. She has the same issues on her end as well. Both phones were already restarted and it didn’t fix the problem,"@143489 Thanks so much for reaching back out! So we can look closer into this with you, reach out to us in DM using the link below and we'll continue from there. https://t.co/GDrqU22YpT"
94,95,"Overnight my messages in my iPhones be have changed and are not recognising my contacts?? What’s going on Please advise, this is ridiculous the amount of issues happening!","@142193 Thanks for reaching out. We'd like some additional information to help. Are the messages showing out of chronological order? Also, when you state your contacts aren't recognized are you missing contact information? Let us know the details in a DM and we'll continue. https://t.co/GDrqU22YpT"
95,96,Is it still possible to install IOS 10.3.3 on iPhone 6s?,@140656 You should be able to update your iPhone to the latest version of iOS 11 actually. Find out more online here: https://t.co/80YRnjDFDk DM us if you have any questions. https://t.co/GDrqU22YpT
96,97,A few hours ago...I’m on WiFi at a restaurant cellular connections not good enough,@136594 Are you able to access other Internet-based content while connected to the restaurant Wi-Fi?
97,98,"Can't update iPhone software, restarted phone and reconnected internet still not working",@127218 Let's look into this together. Which model of iPhone are you using? Here's a great guide to help: https://t.co/Ui1yTnOCUz
98,99,Just received new unit of iPhone X. First had horrible yellow screen and this is even worse. Is gonna finally admit that the some screens are not as good as they claim to be? Apple iPhoneX,@135591 Congratulations on your new iPhone! Let's get to the bottom of what's going on. Please send us a DM with your country and we can get started there. https://t.co/GDrqU22YpT
99,100,IOS new update have some problems. Have to charge the phone 3 times a day. Before it was just once.,@127404 We understand how important battery life is. What is the exact version you updated to? https://t.co/ZTw54HL4Rm


In [48]:
# Cell 39 — Add human-validated labels for Cluster 9

cluster_9_labels = {
    91: ("other_unclear", "high",
         "Backup failure is not covered by the current taxonomy, and the message does not identify a specific supported intent."),

    92: ("performance_freezing", "high",
         "Customer explicitly reports the iPhone becoming slower and freezing after an update."),

    93: ("other_unclear", "high",
         "Alarm failure is a specific issue but does not fit the current 10-intent taxonomy."),

    94: ("connectivity", "medium",
         "FaceTime is failing on both phones even after restart, suggesting a communication/connectivity problem, although the exact cause is unclear."),

    95: ("other_unclear", "high",
         "Messages and contacts are behaving incorrectly, but this does not clearly fit the current taxonomy."),

    96: ("software_update", "high",
         "Customer is asking whether iOS 10.3.3 can be installed, making software version/update the main topic."),

    97: ("connectivity", "high",
         "Customer specifically mentions Wi-Fi and cellular connectivity problems."),

    98: ("device_hardware", "high",
         "Customer reports a physical display/screen quality problem on a newly received iPhone."),

    99: ("battery_power", "high",
         "Battery usage increased from once to three charges per day after the update."),

    100: ("battery_power", "high",
         "Customer explicitly reports needing to charge the phone three times a day after the new iOS update.")
}

for annotation_id, (intent, confidence, note) in cluster_9_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id

    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 9
cluster_9_check = annotation_df[
    annotation_df["semantic_cluster"] == 9
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_9_check)

print(
    "Missing labels in Cluster 9:",
    cluster_9_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
90,91,other_unclear,high,"Backup failure is not covered by the current taxonomy, and the message does not identify a specific supported intent."
91,92,performance_freezing,high,Customer explicitly reports the iPhone becoming slower and freezing after an update.
92,93,other_unclear,high,Alarm failure is a specific issue but does not fit the current 10-intent taxonomy.
93,94,connectivity,medium,"FaceTime is failing on both phones even after restart, suggesting a communication/connectivity problem, although the exact cause is unclear."
94,95,other_unclear,high,"Messages and contacts are behaving incorrectly, but this does not clearly fit the current taxonomy."
95,96,software_update,high,"Customer is asking whether iOS 10.3.3 can be installed, making software version/update the main topic."
96,97,connectivity,high,Customer specifically mentions Wi-Fi and cellular connectivity problems.
97,98,device_hardware,high,Customer reports a physical display/screen quality problem on a newly received iPhone.
98,99,battery_power,high,Battery usage increased from once to three charges per day after the update.
99,100,battery_power,high,Customer explicitly reports needing to charge the phone three times a day after the new iOS update.


Missing labels in Cluster 9: 0


In [49]:
# Cell 40 — Inspect Cluster 10 messages

cluster_10 = annotation_df[
    annotation_df["semantic_cluster"] == 10
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_10)


,annotation_id,customer_message,support_response
100,101,why is my phone acting up saying im out of storage? How do I go from 60gb then delete Spotify(3gb) and end up with 59gb,"@128495 Let's work on this together. When you get the message about storage, is it referring to your iCloud Storage?"
101,102,"Ok, I have a $4.99 student account w/Apple Music but just realized some days ago they've been charging me da 12.99 family plan @115858 fix it",@128695 Thanks for reaching out. For billing issues with Apple Music reach out to our iTunes Store Support: https://t.co/SDIe7UiyJN
102,103,how do I cancel my Apple music trial there is no option to?,"@135646 Check out this article, it will provide the steps to cancel a subscription: https://t.co/YKDbvueXrq"
103,104,I keep getting this error message saying “the item can’t be played” and idk how to fix it...,@116348 We'd like to look into this with you. Does the song appear to be greyed out?
104,105,"apple music got renewed a couple of days ago, but unable to access it right now",@127594 We want to help you get back to enjoying your Apple Music! Which device are you accessing it on? Do you get an error message?
105,106,"Pandora was playing, then suddenly the apple logo appeared and Pandora stopped and a few seconds later I had to type in my passcode to unlock the phone. I found all my open apps backgrounded but I had to forground Pandora and it started playing again.",@142505 Are all of your apps up-to-date if you connect to Wi-Fi and check in its App Store &gt; Updates section?\n\nHow many times has this occurred?
106,107,"And all of a sudden, for no reason, 80GB of music got wiped. WTF, ? ›","@128489 We know how important your music is, and we'd like to look into what happened to your songs. To start, send us a DM. https://t.co/GDrqU22YpT"
107,108,"All website. I can record with audio. The recording are playing well in Photos and the sound is ok. But when I import them in iMovie, the sound doesn’t follow. Only if I was in Safari when I made the screen recording",@121032 Is this iMovie for iOS or for the Mac?
108,109,It’s says it’s “indexing” and won’t play my music,@128488 Thanks for that info. Restart the iPhone and then DM us if that issue persists. https://t.co/GDrqU22YpT
109,110,Apple Music?? That way I've lost a lot of unnecessary money and I still can't listen to albums and single offline or most of the time even with WIFI because it keeps telling me I need an Apple Music plan for my ITUNES music?? please help,@136038 We'd like to see what's going on with your music you purchase. DM us and let us know which device you are purchasing the music on and if you have iCloud Music Library enabled. We'll get started there. https://t.co/GDrqU2kzhr


In [50]:
# Cell 41 — Add human-validated labels for Cluster 10

cluster_10_labels = {
    101: ("other_unclear", "high",
          "Storage-management problem does not clearly fit the current taxonomy; the message does not establish that iCloud/account access is the main issue."),

    102: ("apple_music", "high",
          "Customer is reporting an Apple Music subscription and billing issue."),

    103: ("apple_music", "high",
          "Customer wants to cancel an Apple Music trial."),

    104: ("apple_music", "high",
          "The customer reports that a music item cannot be played, making this a music playback problem."),

    105: ("apple_music", "high",
          "Customer's Apple Music subscription was renewed but the service cannot be accessed."),

    106: ("performance_freezing", "high",
          "The phone appears to restart or crash unexpectedly, causing apps to be backgrounded and requiring the passcode again."),

    107: ("apps_app_store", "high",
          "The problem occurs when importing a recording into iMovie, making it an application-specific issue."),

    108: ("apple_music", "high",
          "Music cannot play because Apple Music is stuck on 'indexing'."),

    109: ("apple_music", "high",
          "Customer cannot play music or albums offline and reports an Apple Music/iTunes access problem."),

    110: ("apple_music", "high",
          "Customer explicitly reports Apple Music/iTunes music access and subscription-related problems.")
}

for annotation_id, (intent, confidence, note) in cluster_10_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id

    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 10
cluster_10_check = annotation_df[
    annotation_df["semantic_cluster"] == 10
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_10_check)

print(
    "Missing labels in Cluster 10:",
    cluster_10_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
100,101,other_unclear,high,Storage-management problem does not clearly fit the current taxonomy; the message does not establish that iCloud/account access is the main issue.
101,102,apple_music,high,Customer is reporting an Apple Music subscription and billing issue.
102,103,apple_music,high,Customer wants to cancel an Apple Music trial.
103,104,apple_music,high,"The customer reports that a music item cannot be played, making this a music playback problem."
104,105,apple_music,high,Customer's Apple Music subscription was renewed but the service cannot be accessed.
105,106,performance_freezing,high,"The phone appears to restart or crash unexpectedly, causing apps to be backgrounded and requiring the passcode again."
106,107,apps_app_store,high,"The problem occurs when importing a recording into iMovie, making it an application-specific issue."
107,108,apple_music,high,Music cannot play because Apple Music is stuck on 'indexing'.
108,109,apple_music,high,Customer cannot play music or albums offline and reports an Apple Music/iTunes access problem.
109,110,apple_music,high,Customer explicitly reports Apple Music/iTunes music access and subscription-related problems.


Missing labels in Cluster 10: 0


In [51]:
# Cell 42 — Inspect Cluster 11 messages

cluster_11 = annotation_df[
    annotation_df["semantic_cluster"] == 11
][
    [
        "annotation_id",
        "customer_message",
        "support_response"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(cluster_11)

,annotation_id,customer_message,support_response
110,111,I can’t because it sends a code to the phone I can’t use??,"@125905 Do you have access to a computer? If so, you can log into your Apple ID account page to make these changes shown in the guide."
111,112,Are refurbished iPads used less than a month? 30 day return policy from most stores!!,"@140207 Hello and thanks for reaching out to us. We test and certify all Apple refurbished products and include a 1-year warranty. All refurbished iPad models also include a brand new battery, new outer shell and a new white box. DM us with any other questions. https://t.co/GDrqU22YpT"
112,113,Thank you. Twitter didn’t notify me of your reply :( so restored phone & ok now. Was phone storage not iCloud.,"@125426 We're glad to hear the issue has since been resolved. Let us know if you have any further questions. Additionally, there may have been a new update to iOS 11. If you're not on iOS 11.1.1, we'd recommend backing up and updating to that latest version."
113,114,I’m having trouble accessing your website to purchase something. Any chance you know when it’ll be available again?,@143285 We're happy to help. What happens when you try to make a purchase? Are you getting an error message?
114,115,"No, these are all the options in the general spot. Siri settings also do not appear in search.","@144223 You mentioned wanting to use Siri only on your Apple Watch, is that right? To be sure we understand, do you have Siri disabled on the paired iPhone right now?"
115,116,Four videos are suddenly gone from my phone...would the automatic iCloud backing-up do that? Clear space by transferring them there?? That's the only thing that could make any sense.,@135614 Let's check some settings on your device. Tap Settings &gt; Photos. Is iCloud Photo Library on? Have you signed out of your Apple ID?
116,117,where do we forward fake Apple emails?,@139840 We certainly appreciate you reaching out to us on this. This article should assist: https://t.co/LNMCdqt6fD
117,118,Hey - Over the last few days my messages app on my iMac and MBA stopped syncing with my iMessages from my iOS devices. Any ideas on how to fix this problem? I've lost one of my favorite parts of AppleSimplicity,"@138696 We’d love to help with your messages. To start, do you see your iMac and MacBook enabled if you tap Settings &gt; Messages &gt; Text Message Forwarding on your iPhone?"
118,119,I'm desperate. my grandkid just turned on voiceover with the internet off. I tried rebooting and now it wants my passcode but won't let me enter it. How do I get access to my ipad again.,@142196 We've got your DM and will continue with you there! Be on the lookout for our reply. Thank you!
119,120,"Yes. The same email accounts are on my Mac, iPad, and 6S Plus",@143490 Your mail should sync automatically. Are you experiencing anything different with your devices?


In [52]:
# Cell 43 — Add human-validated labels for Cluster 11

cluster_11_labels = {
    111: ("apple_id_icloud", "high",
          "Customer cannot receive the verification code because they cannot access the trusted phone, which is an Apple ID/account-access problem."),

    112: ("other_unclear", "high",
          "Customer is asking about refurbished iPad usage and return policy, which is outside the technical-support taxonomy."),

    113: ("other_unclear", "high",
          "Customer is asking about phone storage and distinguishes it from iCloud, but the actual storage issue is outside the current taxonomy."),

    114: ("other_unclear", "high",
          "Customer cannot access Apple's website to make a purchase; website and purchasing issues are outside the current taxonomy."),

    115: ("other_unclear", "high",
          "Siri settings and Apple Watch configuration do not fit any of the current 10 intents."),

    116: ("apple_id_icloud", "high",
          "Customer asks about iCloud automatically backing up or removing photos and mentions Apple ID settings."),

    117: ("other_unclear", "high",
          "Reporting fake Apple emails is a security and phishing-reporting issue, which is outside the current taxonomy."),

    118: ("connectivity", "medium",
          "iMessage is not syncing between devices, indicating a device-to-device communication or synchronization problem, although the exact cause is unclear."),

    119: ("other_unclear", "high",
          "VoiceOver and passcode access problem does not clearly fit the current taxonomy."),

    120: ("other_unclear", "high",
          "Customer only confirms that the same email accounts exist across devices; no specific support problem is stated.")
}

for annotation_id, (intent, confidence, note) in cluster_11_labels.items():
    mask = annotation_df["annotation_id"] == annotation_id

    annotation_df.loc[mask, "human_intent"] = intent
    annotation_df.loc[mask, "annotation_confidence"] = confidence
    annotation_df.loc[mask, "annotation_notes"] = note

# Save final annotation progress
annotation_df.to_csv(
    "../data/processed/human_annotation_progress.csv",
    index=False
)

# Verify Cluster 11
cluster_11_check = annotation_df[
    annotation_df["semantic_cluster"] == 11
][
    [
        "annotation_id",
        "human_intent",
        "annotation_confidence",
        "annotation_notes"
    ]
]

display(cluster_11_check)

print(
    "Missing labels in Cluster 11:",
    cluster_11_check["human_intent"].eq("").sum()
)

,annotation_id,human_intent,annotation_confidence,annotation_notes
110,111,apple_id_icloud,high,"Customer cannot receive the verification code because they cannot access the trusted phone, which is an Apple ID/account-access problem."
111,112,other_unclear,high,"Customer is asking about refurbished iPad usage and return policy, which is outside the technical-support taxonomy."
112,113,other_unclear,high,"Customer is asking about phone storage and distinguishes it from iCloud, but the actual storage issue is outside the current taxonomy."
113,114,other_unclear,high,Customer cannot access Apple's website to make a purchase; website and purchasing issues are outside the current taxonomy.
114,115,other_unclear,high,Siri settings and Apple Watch configuration do not fit any of the current 10 intents.
115,116,apple_id_icloud,high,Customer asks about iCloud automatically backing up or removing photos and mentions Apple ID settings.
116,117,other_unclear,high,"Reporting fake Apple emails is a security and phishing-reporting issue, which is outside the current taxonomy."
117,118,connectivity,medium,"iMessage is not syncing between devices, indicating a device-to-device communication or synchronization problem, although the exact cause is unclear."
118,119,other_unclear,high,VoiceOver and passcode access problem does not clearly fit the current taxonomy.
119,120,other_unclear,high,Customer only confirms that the same email accounts exist across devices; no specific support problem is stated.


Missing labels in Cluster 11: 0


In [53]:
# Cell 44 — Final Human Annotation Summary

# Check that all 120 annotations have labels
missing_labels = annotation_df["human_intent"].eq("").sum()

print("Total annotations:", len(annotation_df))
print("Missing intent labels:", missing_labels)
print(
    "Annotation completion:",
    f"{(1 - missing_labels / len(annotation_df)) * 100:.1f}%"
)

# Intent distribution
intent_distribution = (
    annotation_df["human_intent"]
    .value_counts()
    .rename_axis("human_intent")
    .reset_index(name="count")
)

intent_distribution["percentage"] = (
    intent_distribution["count"] / len(annotation_df) * 100
).round(2)

display(intent_distribution)

Total annotations: 120
Missing intent labels: 0
Annotation completion: 100.0%


,human_intent,count,percentage
0,other_unclear,49,40.83
1,keyboard_input,13,10.83
2,battery_power,13,10.83
3,performance_freezing,9,7.50
4,software_update,8,6.67
5,apps_app_store,7,5.83
6,apple_music,7,5.83
7,apple_id_icloud,5,4.17
8,connectivity,5,4.17
9,device_hardware,4,3.33


In [54]:
# Cell 45 — Compare Semantic Clusters with Human-Validated Intents

cluster_intent_table = pd.crosstab(
    annotation_df["semantic_cluster"],
    annotation_df["human_intent"]
)

display(cluster_intent_table)

human_intent,apple_id_icloud,apple_music,apps_app_store,battery_power,connectivity,device_hardware,keyboard_input,other_unclear,performance_freezing,software_update
semantic_cluster,,,,,,,,,,
0,0,0,2,0,0,0,1,3,1,3
1,0,0,0,0,0,0,0,10,0,0
2,0,0,0,0,0,0,0,9,0,1
3,0,0,0,0,0,0,10,0,0,0
4,0,0,4,0,0,0,0,3,1,2
5,1,0,0,1,1,1,0,4,2,0
6,2,0,0,0,1,1,0,4,2,0
7,0,0,0,9,0,0,0,0,1,0
8,0,0,0,1,0,1,2,5,0,1


In [55]:
# Cell 46 — Dominant Intent per Semantic Cluster

cluster_dominant_intent = (
    annotation_df
    .groupby("semantic_cluster")["human_intent"]
    .agg(
        dominant_intent=lambda x: x.value_counts().index[0],
        dominant_count=lambda x: x.value_counts().iloc[0],
        total_examples="count"
    )
    .reset_index()
)

cluster_dominant_intent["dominant_percentage"] = (
    cluster_dominant_intent["dominant_count"]
    / cluster_dominant_intent["total_examples"]
    * 100
).round(2)

display(cluster_dominant_intent)

,semantic_cluster,dominant_intent,dominant_count,total_examples,dominant_percentage
0,0,other_unclear,3,10,30.0
1,1,other_unclear,10,10,100.0
2,2,other_unclear,9,10,90.0
3,3,keyboard_input,10,10,100.0
4,4,apps_app_store,4,10,40.0
5,5,other_unclear,4,10,40.0
6,6,other_unclear,4,10,40.0
7,7,battery_power,9,10,90.0
8,8,other_unclear,5,10,50.0
9,9,other_unclear,3,10,30.0


In [56]:
# Cell 47 — Semantic Cluster Consistency

cluster_consistency = (
    annotation_df
    .groupby("semantic_cluster")["human_intent"]
    .apply(
        lambda x: x.value_counts(normalize=True).iloc[0]
    )
    .reset_index(name="consistency")
)

cluster_consistency["consistency_percentage"] = (
    cluster_consistency["consistency"] * 100
).round(2)

display(cluster_consistency)

print(
    "Average cluster consistency:",
    round(cluster_consistency["consistency"].mean() * 100, 2),
    "%"
)

,semantic_cluster,consistency,consistency_percentage
0,0,0.3,30.0
1,1,1.0,100.0
2,2,0.9,90.0
3,3,1.0,100.0
4,4,0.4,40.0
5,5,0.4,40.0
6,6,0.4,40.0
7,7,0.9,90.0
8,8,0.5,50.0
9,9,0.3,30.0


Average cluster consistency: 62.5 %


In [57]:
# Cell 48 — Create Final Human-Validated Dataset

human_validated_df = annotation_df[
    [
        "annotation_id",
        "customer_message",
        "human_intent",
        "annotation_confidence",
        "annotation_notes",
        "semantic_cluster"
    ]
].copy()

# Rename columns for the modeling pipeline
human_validated_df = human_validated_df.rename(
    columns={
        "customer_message": "text",
        "human_intent": "intent"
    }
)

# Remove accidental whitespace
human_validated_df["text"] = (
    human_validated_df["text"]
    .astype(str)
    .str.strip()
)

human_validated_df["intent"] = (
    human_validated_df["intent"]
    .astype(str)
    .str.strip()
)

# Verify
print("Dataset shape:", human_validated_df.shape)
print(
    "Missing text:",
    human_validated_df["text"].eq("").sum()
)
print(
    "Missing intent:",
    human_validated_df["intent"].eq("").sum()
)

display(human_validated_df.head())

# Save
human_validated_df.to_csv(
    "../data/processed/human_validated_intents.csv",
    index=False
)

print("\nSaved:")
print("../data/processed/human_validated_intents.csv")

Dataset shape: (120, 6)
Missing text: 0
Missing intent: 0


,annotation_id,text,intent,annotation_confidence,annotation_notes,semantic_cluster
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,high,Delete key on Bluetooth keyboard stopped working while typing.,0
1,2,I am using an iPhone 8 Plus and iOS 11.1.2,other_unclear,high,Only provides device and iOS version; no problem is stated.,0
2,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",performance_freezing,high,Phone repeatedly freezes; camera and gallery problems are secondary.,0
3,4,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,software_update,high,No sound started immediately after the iOS update.,0
4,5,"About 2 weeks, I have IOS 11.1.2",other_unclear,high,Only states the iOS version and duration; no clear support issue.,0



Saved:
../data/processed/human_validated_intents.csv


In [58]:
# Cell 49 — Validate the final human-labeled dataset

print("Total samples:", len(human_validated_df))
print("Unique intents:", human_validated_df["intent"].nunique())

print("\nIntent distribution:")
display(
    human_validated_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

print("\nConfidence distribution:")
display(
    human_validated_df["annotation_confidence"]
    .value_counts()
    .rename_axis("confidence")
    .reset_index(name="count")
)

print("\nDuplicate texts:")
print(
    human_validated_df["text"].duplicated().sum()
)

Total samples: 120
Unique intents: 10

Intent distribution:


,intent,count
0,other_unclear,49
1,keyboard_input,13
2,battery_power,13
3,performance_freezing,9
4,software_update,8
5,apps_app_store,7
6,apple_music,7
7,apple_id_icloud,5
8,connectivity,5
9,device_hardware,4



Confidence distribution:


,confidence,count
0,high,107
1,medium,13



Duplicate texts:
1


In [59]:
# Cell 50 — Check class sizes before model evaluation

class_counts = human_validated_df["intent"].value_counts()

print("Minimum class size:", class_counts.min())

display(
    class_counts
    .sort_values()
    .rename_axis("intent")
    .reset_index(name="count")
)

Minimum class size: 4


,intent,count
0,device_hardware,4
1,apple_id_icloud,5
2,connectivity,5
3,apps_app_store,7
4,apple_music,7
5,software_update,8
6,performance_freezing,9
7,keyboard_input,13
8,battery_power,13
9,other_unclear,49


In [60]:
# Cell 51 — TF-IDF + Logistic Regression Baseline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

X = human_validated_df["text"]
y = human_validated_df["intent"]

# TF-IDF + Logistic Regression pipeline
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# 4-fold stratified CV
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

scoring = [
    "accuracy",
    "f1_macro",
    "f1_weighted"
]

cv_results = cross_validate(
    baseline_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

print("TF-IDF + Logistic Regression")
print("--------------------------------")

for metric in scoring:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

TF-IDF + Logistic Regression
--------------------------------
accuracy: 0.3667 +/- 0.1027
f1_macro: 0.2124 +/- 0.0479
f1_weighted: 0.3470 +/- 0.0842


In [61]:
# Cell 52 — Generate Out-of-Fold Predictions

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions using the same 4-fold CV
oof_predictions = cross_val_predict(
    baseline_model,
    X,
    y,
    cv=cv,
    method="predict"
)

# Classification report
print("TF-IDF + Logistic Regression — Out-of-Fold Evaluation")
print("=" * 60)

print(
    classification_report(
        y,
        oof_predictions,
        zero_division=0
    )
)

# Confusion matrix
labels = sorted(y.unique())

cm = confusion_matrix(
    y,
    oof_predictions,
    labels=labels
)

confusion_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(confusion_df)

TF-IDF + Logistic Regression — Out-of-Fold Evaluation
                      precision    recall  f1-score   support

     apple_id_icloud       0.00      0.00      0.00         5
         apple_music       0.45      0.71      0.56         7
      apps_app_store       0.00      0.00      0.00         7
       battery_power       0.33      0.62      0.43        13
        connectivity       0.00      0.00      0.00         5
     device_hardware       0.00      0.00      0.00         4
      keyboard_input       0.29      0.54      0.38        13
       other_unclear       0.59      0.45      0.51        49
performance_freezing       0.17      0.22      0.19         9
     software_update       0.00      0.00      0.00         8

            accuracy                           0.37       120
           macro avg       0.18      0.25      0.21       120
        weighted avg       0.35      0.37      0.34       120



,apple_id_icloud,apple_music,apps_app_store,battery_power,connectivity,device_hardware,keyboard_input,other_unclear,performance_freezing,software_update
apple_id_icloud,0,1,0,1,0,0,1,2,0,0
apple_music,1,5,0,0,0,0,0,0,0,1
apps_app_store,0,1,0,2,0,0,1,2,1,0
battery_power,0,0,0,8,0,0,1,3,0,1
connectivity,0,1,0,2,0,0,1,0,1,0
device_hardware,0,0,0,0,0,0,1,1,2,0
keyboard_input,0,0,0,0,0,0,7,3,2,1
other_unclear,1,2,2,7,0,0,9,22,3,3
performance_freezing,0,1,0,2,0,1,0,3,2,0
software_update,1,0,0,2,0,0,3,1,1,0


In [63]:
# Cell 53 — Load Saved Customer Embeddings

import numpy as np

customer_embeddings = np.load(
    "../data/embeddings/customer_embeddings.npy"
)

print("Embedding shape:", customer_embeddings.shape)
print("Number of labeled samples:", len(human_validated_df))

Embedding shape: (3092, 384)
Number of labeled samples: 120


In [66]:
# Cell 55 — Sentence Embeddings + Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Select embeddings for the 120 human-validated examples
valid_mask = human_validated_df["embedding_index"].notna()

X_embeddings = customer_embeddings[
    human_validated_df.loc[valid_mask, "embedding_index"].astype(int)
]

y_embeddings = human_validated_df.loc[
    valid_mask, "intent"
]

print("Embedding matrix:", X_embeddings.shape)
print("Labels:", y_embeddings.shape)

# Classifier
embedding_classifier = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

# Same CV strategy as the TF-IDF baseline
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

scoring = [
    "accuracy",
    "f1_macro",
    "f1_weighted"
]

embedding_cv_results = cross_validate(
    embedding_classifier,
    X_embeddings,
    y_embeddings,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

print("\nSentence Embeddings + Logistic Regression")
print("------------------------------------------")

for metric in scoring:
    scores = embedding_cv_results[f"test_{metric}"]
    print(
        f"{metric}: "
        f"{scores.mean():.4f} +/- {scores.std():.4f}"
    )

Embedding matrix: (120, 384)
Labels: (120,)

Sentence Embeddings + Logistic Regression
------------------------------------------
accuracy: 0.5000 +/- 0.0408
f1_macro: 0.3411 +/- 0.0474
f1_weighted: 0.5002 +/- 0.0537


In [68]:
# Cell 57 — Inspect Embedding Model Errors

error_df = human_validated_df.loc[
    valid_mask,
    ["annotation_id", "text", "intent"]
].copy()

error_df["predicted_intent"] = embedding_oof_predictions

# Only incorrect predictions
errors = error_df[
    error_df["intent"] != error_df["predicted_intent"]
].copy()

print("Total validation examples:", len(error_df))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    f"{len(errors) / len(error_df):.2%}"
)

print("\nMost common wrong predictions")
print("--------------------------------")

error_summary = (
    errors
    .groupby(["intent", "predicted_intent"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(error_summary)

print("\nExamples from failed intents")
print("--------------------------------")

failed_intents = [
    "apple_id_icloud",
    "apps_app_store",
    "device_hardware",
    "software_update"
]

display(
    errors[
        errors["intent"].isin(failed_intents)
    ][
        ["annotation_id", "text", "intent", "predicted_intent"]
    ].to_string(index=False)
)

Total validation examples: 120
Incorrect predictions: 60
Error rate: 50.00%

Most common wrong predictions
--------------------------------


,intent,predicted_intent,count
20,other_unclear,apps_app_store,4
23,other_unclear,keyboard_input,4
25,other_unclear,software_update,4
19,other_unclear,apple_id_icloud,4
21,other_unclear,battery_power,3
4,apps_app_store,apple_music,3
24,other_unclear,performance_freezing,3
1,apple_id_icloud,device_hardware,2
35,software_update,other_unclear,2
34,software_update,device_hardware,2



Examples from failed intents
--------------------------------


" annotation_id                                                                                                                                                                                                                                                                       text          intent     predicted_intent\n             4                                                                                                                                                                                                  Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes software_update       apps_app_store\n             6                                                                                                                                                                                                                             I got a crazy security bug issue with iOS 11.1 software_update        battery_power\n             7                         

In [69]:
# Cell 58 — Sentence Embeddings + Linear SVM

from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate

svm_classifier = LinearSVC(
    class_weight="balanced",
    C=1.0,
    random_state=42
)

scoring = [
    "accuracy",
    "f1_macro",
    "f1_weighted"
]

svm_cv_results = cross_validate(
    svm_classifier,
    X_embeddings,
    y_embeddings,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

print("Sentence Embeddings + Linear SVM")
print("--------------------------------")

for metric in scoring:
    scores = svm_cv_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} +/- {scores.std():.4f}"
    )

Sentence Embeddings + Linear SVM
--------------------------------
accuracy: 0.5417 +/- 0.0759
f1_macro: 0.3176 +/- 0.0410
f1_weighted: 0.5055 +/- 0.0805


In [70]:
# Cell 58 — Sentence Embeddings + Linear SVM

from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate

svm_classifier = LinearSVC(
    class_weight="balanced",
    C=1.0,
    random_state=42
)

scoring = [
    "accuracy",
    "f1_macro",
    "f1_weighted"
]

svm_cv_results = cross_validate(
    svm_classifier,
    X_embeddings,
    y_embeddings,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

print("Sentence Embeddings + Linear SVM")
print("--------------------------------")

for metric in scoring:
    scores = svm_cv_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} +/- {scores.std():.4f}"
    )

Sentence Embeddings + Linear SVM
--------------------------------
accuracy: 0.5417 +/- 0.0759
f1_macro: 0.3176 +/- 0.0410
f1_weighted: 0.5055 +/- 0.0805


In [72]:
# Cell 59 — Confidence Analysis for Safe Routing

from sklearn.model_selection import cross_val_predict
import numpy as np
import pandas as pd

# Out-of-fold probability predictions
embedding_oof_probabilities = cross_val_predict(
    embedding_classifier,
    X_embeddings,
    y_embeddings,
    cv=cv,
    method="predict_proba"
)

# The class order used by Logistic Regression
# follows the sorted unique labels
class_labels = np.sort(y_embeddings.unique())

# Predicted class
embedding_oof_pred = class_labels[
    np.argmax(embedding_oof_probabilities, axis=1)
]

# Highest probability = model confidence
prediction_confidence = np.max(
    embedding_oof_probabilities,
    axis=1
)

confidence_df = pd.DataFrame({
    "true_intent": y_embeddings.values,
    "predicted_intent": embedding_oof_pred,
    "confidence": prediction_confidence
})

confidence_df["correct"] = (
    confidence_df["true_intent"]
    == confidence_df["predicted_intent"]
)

print("Overall confidence statistics")
print("--------------------------------")

display(
    confidence_df["confidence"].describe()
)

print("\nConfidence by prediction correctness")
print("--------------------------------------")

display(
    confidence_df.groupby("correct")["confidence"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nConfidence buckets")
print("------------------")

confidence_df["confidence_bucket"] = pd.cut(
    confidence_df["confidence"],
    bins=[0, 0.50, 0.70, 0.85, 1.00],
    labels=[
        "<= 0.50",
        "0.50-0.70",
        "0.70-0.85",
        "0.85-1.00"
    ],
    include_lowest=True
)

display(
    confidence_df.groupby(
        "confidence_bucket",
        observed=True
    )["correct"]
    .agg(["count", "mean"])
)

Overall confidence statistics
--------------------------------


count    120.000000
mean       0.196097
std        0.064693
min        0.120579
25%        0.150696
50%        0.173691
75%        0.233938
max        0.437023
Name: confidence, dtype: float64


Confidence by prediction correctness
--------------------------------------


,count,mean,median,min,max
correct,,,,,
False,60,0.158185,0.157095,0.120579,0.223152
True,60,0.234009,0.234407,0.132960,0.437023



Confidence buckets
------------------


,count,mean
confidence_bucket,,
<= 0.50,120,0.5


In [73]:
# Cell 60 — Confidence Threshold Analysis

thresholds = [
    0.15,
    0.18,
    0.20,
    0.22,
    0.25,
    0.30,
    0.35,
    0.40
]

threshold_results = []

for threshold in thresholds:

    accepted = confidence_df[
        confidence_df["confidence"] >= threshold
    ]

    if len(accepted) == 0:
        continue

    coverage = len(accepted) / len(confidence_df)

    accuracy = accepted["correct"].mean()

    threshold_results.append({
        "threshold": threshold,
        "accepted_samples": len(accepted),
        "coverage": coverage,
        "accuracy_on_accepted": accuracy
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,threshold,accepted_samples,coverage,accuracy_on_accepted
0,0.15,91,0.758333,0.593407
1,0.18,53,0.441667,0.811321
2,0.20,41,0.341667,0.926829
3,0.22,35,0.291667,0.971429
4,0.25,23,0.191667,1.000000
5,0.30,8,0.066667,1.000000
6,0.35,5,0.041667,1.000000
7,0.40,2,0.016667,1.000000


In [74]:
# Cell 61 — Train Final Intent Classifier

from sklearn.linear_model import LogisticRegression
import joblib

# Train on all human-validated examples
final_intent_classifier = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

final_intent_classifier.fit(
    X_embeddings,
    y_embeddings
)

print("Final classifier trained successfully.")
print("Training samples:", len(y_embeddings))
print("Number of intents:", len(final_intent_classifier.classes_))
print("Intents:")
print(list(final_intent_classifier.classes_))

Final classifier trained successfully.
Training samples: 120
Number of intents: 10
Intents:
['apple_id_icloud', 'apple_music', 'apps_app_store', 'battery_power', 'connectivity', 'device_hardware', 'keyboard_input', 'other_unclear', 'performance_freezing', 'software_update']


In [75]:
# Cell 62 — Save Intent Routing Configuration

import json

routing_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "classifier": "LogisticRegression",
    "training_samples": 120,
    "num_intents": 10,
    "confidence_threshold": 0.20,
    "decision_policy": {
        "high_confidence": "Use predicted intent for retrieval",
        "low_confidence": "Ask clarification or escalate to human"
    },
    "evaluation": {
        "accuracy": 0.5000,
        "macro_f1": 0.3411,
        "weighted_f1": 0.5002,
        "evaluation_method": "4-fold stratified cross-validation"
    }
}

with open(
    "../models/routing_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        routing_config,
        f,
        indent=2
    )

print("Saved routing configuration:")
print("../models/routing_config.json")

Saved routing configuration:
../models/routing_config.json


In [78]:
# Cell 63 — Verify Saved Model Artifacts

import os

model_files = [
    "../models/intent_classifier.joblib",
    "../models/routing_config.json"
]

for file_path in model_files:
    print(
        file_path,
        "->",
        "EXISTS" if os.path.exists(file_path) else "MISSING"
    )

../models/intent_classifier.joblib -> EXISTS
../models/routing_config.json -> EXISTS


In [77]:
# Cell 64 — Save Final Intent Classifier

import joblib
import os

model_path = "../models/intent_classifier.joblib"

joblib.dump(
    final_intent_classifier,
    model_path
)

print("Saved:", model_path)
print("Exists:", os.path.exists(model_path))

Saved: ../models/intent_classifier.joblib
Exists: True


In [79]:
# Cell 66 — Inspect Actual Training Text

print("Examples actually used by the intent classifier")
print("=" * 80)

for i in range(15):
    row = human_validated_df.iloc[i]

    print(f"\nExample {i + 1}")
    print("Intent :", row["intent"])
    print("Text   :", row["text"])

Examples actually used by the intent classifier

Example 1
Intent : keyboard_input
Text   : Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg

Example 2
Intent : other_unclear
Text   : I am using an iPhone 8 Plus and iOS 11.1.2

Example 3
Intent : performance_freezing
Text   : pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?

Example 4
Intent : software_update
Text   : Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes

Example 5
Intent : other_unclear
Text   : About 2 weeks, I have IOS 11.1.2

Example 6
Intent : software_update
Text   : I got a crazy security bug issue with iOS 11.1

Example 7
Intent : software_update
Text   : force touch for app multitask gone in iOS11 ?

Example 8
Intent : apps_app_store
Text   : Hey iOS 11 sucks none of my apps can update

In [80]:
# Cell 67 — Inspect Model Predictions

prediction_check = human_validated_df[
    ["annotation_id", "text", "intent"]
].copy()

prediction_check["predicted_intent"] = embedding_oof_pred
prediction_check["confidence"] = prediction_confidence

prediction_check["correct"] = (
    prediction_check["intent"]
    == prediction_check["predicted_intent"]
)

display(
    prediction_check[
        [
            "annotation_id",
            "intent",
            "predicted_intent",
            "confidence",
            "correct",
            "text"
        ]
    ].head(20)
)

,annotation_id,intent,predicted_intent,confidence,correct,text
0,1,keyboard_input,keyboard_input,0.173898,True,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg
1,2,other_unclear,performance_freezing,0.134179,False,I am using an iPhone 8 Plus and iOS 11.1.2
2,3,performance_freezing,device_hardware,0.173584,False,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?"
3,4,software_update,apps_app_store,0.149632,False,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes
4,5,other_unclear,battery_power,0.156640,False,"About 2 weeks, I have IOS 11.1.2"
5,6,software_update,battery_power,0.128979,False,I got a crazy security bug issue with iOS 11.1
6,7,software_update,apps_app_store,0.137324,False,force touch for app multitask gone in iOS11 ?
7,8,apps_app_store,software_update,0.167843,False,Hey iOS 11 sucks none of my apps can update anymore
8,9,other_unclear,software_update,0.173798,False,ios 11
9,10,apps_app_store,battery_power,0.131001,False,": New iPhoneX. iOS 11.1.2 update. Phone app literally will not open. Cannot access call history, my voicemail or make a call. WTF"


In [81]:
# Cell 68 — Find Similar Messages Across Different Intents

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Similarity between all 120 human-labeled examples
similarity_matrix = cosine_similarity(X_embeddings)

similar_pairs = []

for i in range(len(human_validated_df)):
    for j in range(i + 1, len(human_validated_df)):
        
        intent_i = human_validated_df.iloc[i]["intent"]
        intent_j = human_validated_df.iloc[j]["intent"]
        
        # Only look for similar texts that have DIFFERENT labels
        if intent_i != intent_j:
            similar_pairs.append({
                "annotation_1": human_validated_df.iloc[i]["annotation_id"],
                "intent_1": intent_i,
                "text_1": human_validated_df.iloc[i]["text"],
                "annotation_2": human_validated_df.iloc[j]["annotation_id"],
                "intent_2": intent_j,
                "text_2": human_validated_df.iloc[j]["text"],
                "similarity": similarity_matrix[i, j]
            })

similar_pairs_df = pd.DataFrame(similar_pairs)

similar_pairs_df = similar_pairs_df.sort_values(
    "similarity",
    ascending=False
)

print("Most similar messages with different human labels:")
print("=" * 100)

display(
    similar_pairs_df.head(15)
)

Most similar messages with different human labels:


,annotation_1,intent_1,text_1,annotation_2,intent_2,text_2,similarity
5132,84,software_update,"It’s been a few days since @115858 made me update my phone’s operating system. Now constantly glitching, hmm 🤔 I’m shocked!",86,device_hardware,my phone’s updated tho 🤔 what kinda bug is this @115858 fix it https://t.co/g3DalWEd91,0.785197
5253,88,keyboard_input,@115858 this “I.T” thing gonna be fixed any time soon....,89,other_unclear,@115858 ever since the update I’ve been experience some difficulties #Typical #MylastIphone https://t.co/BeCpD21A2D,0.659856
679,8,apps_app_store,Hey iOS 11 sucks none of my apps can update anymore,9,other_unclear,ios 11,0.651310
765,8,apps_app_store,Hey iOS 11 sucks none of my apps can update anymore,100,battery_power,IOS new update have some problems. Have to charge the phone 3 times a day. Before it was just once.,0.650641
5111,83,other_unclear,@116346 @115858 @115714 Yes a legit nothing fixes it for me,88,keyboard_input,@115858 this “I.T” thing gonna be fixed any time soon....,0.644903
3854,54,connectivity,Soy sólo yo...o a ustedes también les pasa que a toda hora se les enciende el WIFI del teléfono desde la nueva actualización del iOS ?,59,performance_freezing,Desde que actualice a iOS 11 mi iPhone se volvió lento. Me recuerda a los viejos Samsung! Que está pasando?,0.644269
5190,86,device_hardware,my phone’s updated tho 🤔 what kinda bug is this @115858 fix it https://t.co/g3DalWEd91,89,other_unclear,@115858 ever since the update I’ve been experience some difficulties #Typical #MylastIphone https://t.co/BeCpD21A2D,0.643113
287,4,software_update,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,8,apps_app_store,Hey iOS 11 sucks none of my apps can update anymore,0.639661
819,9,other_unclear,ios 11,74,battery_power,Even i have the battery drain issues since updating to iOS 11. Not even any better on iOS 11.1,0.634144
5088,82,other_unclear,@126993 @115858 Even this tweet has the error 🙄. Fix it https://t.co/VRGWHCCsQF,88,keyboard_input,@115858 this “I.T” thing gonna be fixed any time soon....,0.633976


In [82]:
# Cell 69 — Per-Intent Performance

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_embeddings,
    embedding_oof_pred,
    labels=sorted(y_embeddings.unique()),
    zero_division=0
)

per_intent_results = pd.DataFrame({
    "intent": sorted(y_embeddings.unique()),
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})

per_intent_results = per_intent_results.sort_values(
    "f1",
    ascending=False
)

display(per_intent_results)

,intent,precision,recall,f1,support
6,keyboard_input,0.684211,1.000000,0.812500,13
1,apple_music,0.600000,0.857143,0.705882,7
3,battery_power,0.578947,0.846154,0.687500,13
7,other_unclear,0.862069,0.510204,0.641026,49
8,performance_freezing,0.333333,0.333333,0.333333,9
4,connectivity,0.200000,0.200000,0.200000,5
0,apple_id_icloud,0.125000,0.200000,0.153846,5
2,apps_app_store,0.000000,0.000000,0.000000,7
5,device_hardware,0.000000,0.000000,0.000000,4
9,software_update,0.000000,0.000000,0.000000,8


In [83]:
# Cell 70 — Find Most Uncertain Examples

uncertain_df = human_validated_df[
    [
        "annotation_id",
        "text",
        "intent"
    ]
].copy()

uncertain_df["predicted_intent"] = embedding_oof_pred
uncertain_df["confidence"] = prediction_confidence

uncertain_df = uncertain_df.sort_values(
    "confidence",
    ascending=True
)

print("Most uncertain human-labeled examples:")
print("=" * 100)

display(
    uncertain_df.head(20)
)

Most uncertain human-labeled examples:


,annotation_id,text,intent,predicted_intent,confidence
42,43,I have not installed any new appliaction. Anyway automatic application updates are enabled so I cannot determine which of the ones that were installed today might be coincident with the exact time frame when the issue appeared.,other_unclear,apps_app_store,0.120579
62,63,I’ve tried trashing Bluetooth prefs - but this didn’t help. Could you please explain how to run the hardware test?,connectivity,battery_power,0.122826
48,49,When i go to upload to social media with a slo-mo or screen shotted picture i am getting a loading circle in the middle of the screen and it never finishes loading. Doesnt allow upload to complete,software_update,device_hardware,0.125663
69,70,"Update to latest version,did the restart thing but still no vibrate at all when notifications coming",other_unclear,software_update,0.126940
5,6,I got a crazy security bug issue with iOS 11.1,software_update,battery_power,0.128979
110,111,I can’t because it sends a code to the phone I can’t use??,apple_id_icloud,device_hardware,0.130354
9,10,": New iPhoneX. iOS 11.1.2 update. Phone app literally will not open. Cannot access call history, my voicemail or make a call. WTF",apps_app_store,battery_power,0.131001
65,66,Once I removed two step authentication it all went back to normal - thanks for responding,apple_id_icloud,other_unclear,0.132526
96,97,A few hours ago...I’m on WiFi at a restaurant cellular connections not good enough,connectivity,connectivity,0.132960
117,118,Hey - Over the last few days my messages app on my iMac and MBA stopped syncing with my iMessages from my iOS devices. Any ideas on how to fix this problem? I've lost one of my favorite parts of AppleSimplicity,connectivity,apps_app_store,0.133695


In [84]:
# Cell 71 — Inspect Retrieval Dataset

support_pairs = pd.read_csv(
    "../data/processed/support_pairs.csv"
)

print("Total support pairs:", len(support_pairs))
print("\nColumns:")
print(list(support_pairs.columns))

print("\nMissing values:")
display(support_pairs.isna().sum())

print("\nSample retrieval records:")
display(
    support_pairs[
        [
            "customer_message",
            "support_response",
            "response_behavior",
            "response_quality"
        ]
    ].head(10)
)

Total support pairs: 3092

Columns:
['conversation_id', 'customer_tweet_id', 'support_tweet_id', 'customer_message', 'support_response', 'response_behavior', 'response_quality']

Missing values:


conversation_id      0
customer_tweet_id    0
support_tweet_id     0
customer_message     0
support_response     0
response_behavior    0
response_quality     0
dtype: int64


Sample retrieval records:


,customer_message,support_response,response_behavior,response_quality
0,What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!,"@137967 We'd like to help. Does this seem to persist when using multiple apps? Have you tried rebooting to see if that helps? Send us a DM, we'll be glad to work on this with you. https://t.co/GDrqU22YpT",troubleshooting,guided_escalation
1,I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.,"@137967 Either way, let us know what happens via DM. We want to make sure this is fully resolved. https://t.co/GDrqU22YpT",dm_escalation,guided_escalation
2,My iPhone been moving slow af the past couple weeks. I need answers,"@137986 We’d love to help with the performance of your iPhone. To start, please send us a DM confirming which iPhone and exact software version you’re using. You can tap Settings &gt; General &gt; About to locate the software version. https://t.co/GDrqU22YpT",troubleshooting,guided_escalation
3,"Dear god not again,",@138151 We want to make sure your words and texts are coming out as expected. What version of the iOS software do you currently have installed on your device? You can find this in Settings &gt; General &gt; About. \n\nPlease send a DM with this information so we can better assist. https://t.co/GDrqU22YpT,troubleshooting,guided_escalation
4,Noticed a bug on my @115858 iPhone X/iOS 11.1.2 While I’m on the phone I can’t close apps.,"@138152 We can assist you. Thanks for bringing this to our attention. If you restart your device and test the issue again, does it persist?",troubleshooting,actionable
5,"leg dit even uit? Dit is mijn oud e-mail adress/apple id, sinds een recente update krijg ik constant deze melding. Ik heb sinds paar jaar een nieuw e-mail adres en ook een apple id op dit adres. Ik was (nog altijd) het wachtwoord kwijt van het oude e-mail & apple-id",@138153 We offer support via Twitter in English. Contact us for help in your preferred language here: https://t.co/IBIY3vMgPj,link_resource,actionable
6,"Since a recent update i’m constantly getting these notifications about my old apple ID /e-mail adress, wich i’ve lost the pw years ago. I allready got a new account and e-mail, but it doesn’t appear anywhere, alltho i can still acces the app store etc..","@138153 We'd be happy to look into why that Apple ID is popping up now, and see what we need to do to get it to stop. To get started, can you tell us which iPhone and iOS version you're using? You can find the version by going to Settings &gt; General &gt; About &gt; Version.",troubleshooting,actionable
7,"Yes ofcourse, iphone 6s version 11.1.2 (15B202)","@138153 Go ahead and DM us your country, and we'll continue working there. https://t.co/GDrqU22YpT",troubleshooting,guided_escalation
8,@115858 Just because you're releasing new phones doesn't mean you should stop making my current iPhone stop working. I have a contract and cannot just purchase a new phone every 12 months.,"@138154 If at all possible, we would love to help. Can you tell us in more detail about what you're experiencing with the current iPhone you have?",information_request,other
9,anyone else’s phone changing “it” to I.T,@138155 We've received your DM and will respond to you there shortly.,dm_escalation,generic_escalation


In [86]:
# Cell 72 — Create Leakage-Safe Retrieval Split

from sklearn.model_selection import train_test_split

retrieval_df = support_pairs.copy()

# Normalize only for duplicate checking.
# Original customer text remains unchanged.
retrieval_df["text_for_split"] = (
    retrieval_df["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Convert to a normal Python list.
# This avoids the PyArrow indexing error.
unique_messages = (
    retrieval_df["text_for_split"]
    .drop_duplicates()
    .tolist()
)

print("Unique customer messages:", len(unique_messages))

train_messages, eval_messages = train_test_split(
    unique_messages,
    test_size=0.20,
    random_state=42
)

retrieval_train = retrieval_df[
    retrieval_df["text_for_split"].isin(train_messages)
].copy()

retrieval_eval = retrieval_df[
    retrieval_df["text_for_split"].isin(eval_messages)
].copy()

print("\nTotal pairs:", len(retrieval_df))
print("Retrieval corpus:", len(retrieval_train))
print("Evaluation examples:", len(retrieval_eval))

overlap = (
    set(retrieval_train["text_for_split"])
    &
    set(retrieval_eval["text_for_split"])
)

print(
    "Exact normalized customer-message overlap:",
    len(overlap)
)

print("\nRetrieval corpus columns:")
print(list(retrieval_train.columns))

Unique customer messages: 3023

Total pairs: 3092
Retrieval corpus: 2470
Evaluation examples: 622
Exact normalized customer-message overlap: 0

Retrieval corpus columns:
['conversation_id', 'customer_tweet_id', 'support_tweet_id', 'customer_message', 'support_response', 'response_behavior', 'response_quality', 'text_for_split']


In [87]:
# Cell 73 — Create Retrieval Embeddings

from sentence_transformers import SentenceTransformer
import numpy as np

# Reuse the same embedding model
retrieval_embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Original customer messages — no aggressive cleaning
retrieval_train_texts = (
    retrieval_train["customer_message"]
    .astype(str)
    .str.strip()
    .tolist()
)

retrieval_eval_texts = (
    retrieval_eval["customer_message"]
    .astype(str)
    .str.strip()
    .tolist()
)

# Encode retrieval corpus
retrieval_train_embeddings = retrieval_embedding_model.encode(
    retrieval_train_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Encode evaluation queries
retrieval_eval_embeddings = retrieval_embedding_model.encode(
    retrieval_eval_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Retrieval train embeddings:",
      retrieval_train_embeddings.shape)

print("Retrieval evaluation embeddings:",
      retrieval_eval_embeddings.shape)

d:\AgentCustomerSupport\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 20/20 [00:09<00:00,  2.16it/s]

Retrieval train embeddings: (2470, 384)
Retrieval evaluation embeddings: (622, 384)


In [88]:
# Cell 74 — Top-K Retrieval Evaluation

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Calculate similarity between every evaluation query
# and every retrieval-corpus customer message.
similarity_scores = cosine_similarity(
    retrieval_eval_embeddings,
    retrieval_train_embeddings
)

print("Similarity matrix shape:", similarity_scores.shape)

# Evaluate whether retrieved examples have useful similarity.
top_k_values = [1, 3, 5, 10]

retrieval_results = []

for k in top_k_values:
    
    top_k_indices = np.argsort(
        similarity_scores,
        axis=1
    )[:, -k:][:, ::-1]
    
    # Average similarity of the retrieved examples
    top_k_scores = np.take_along_axis(
        similarity_scores,
        top_k_indices,
        axis=1
    )
    
    retrieval_results.append({
        "k": k,
        "mean_similarity": top_k_scores.mean(),
        "median_similarity": np.median(top_k_scores),
        "min_similarity": top_k_scores.min(),
        "max_similarity": top_k_scores.max()
    })

retrieval_results_df = pd.DataFrame(
    retrieval_results
)

display(retrieval_results_df)

Similarity matrix shape: (622, 2470)


,k,mean_similarity,median_similarity,min_similarity,max_similarity
0,1,0.662866,0.653882,0.245530,1.0
1,3,0.629633,0.622983,0.204264,1.0
2,5,0.610707,0.604916,0.193261,1.0
3,10,0.581477,0.574476,0.177749,1.0


In [89]:
# Cell 75 — Retrieval Intent Consistency Evaluation

# Map normalized customer message -> human validated intent
human_intent_map = {
    str(row["text"]).lower().strip(): row["intent"]
    for _, row in human_validated_df.iterrows()
}

# Find human-labeled examples that belong to the retrieval evaluation set
retrieval_eval_labeled = retrieval_eval.copy()

retrieval_eval_labeled["normalized_text"] = (
    retrieval_eval_labeled["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

retrieval_eval_labeled["human_intent"] = (
    retrieval_eval_labeled["normalized_text"]
    .map(human_intent_map)
)

labeled_retrieval_eval = retrieval_eval_labeled[
    retrieval_eval_labeled["human_intent"].notna()
].copy()

print(
    "Human-labeled examples found in retrieval evaluation set:",
    len(labeled_retrieval_eval)
)

Human-labeled examples found in retrieval evaluation set: 28


In [91]:
# Cell 76 — Corrected Top-K Retrieval Intent Consistency

from collections import Counter
import numpy as np

# ---------------------------------------------------------
# 1. Build human intent lookup from the validated dataset
# ---------------------------------------------------------

human_intent_map = {
    str(row["text"]).lower().strip(): row["intent"]
    for _, row in human_validated_df.iterrows()
}

# ---------------------------------------------------------
# 2. Assign human intents to retrieval-train examples
# ---------------------------------------------------------

retrieval_train_intents = []

for _, row in retrieval_train.iterrows():
    text = str(row["customer_message"]).lower().strip()

    retrieval_train_intents.append(
        human_intent_map.get(text, None)
    )

retrieval_train_intents = np.array(
    retrieval_train_intents,
    dtype=object
)

# ---------------------------------------------------------
# 3. Get positions of the 28 labeled evaluation examples
# ---------------------------------------------------------

labeled_eval_positions = retrieval_eval.index.get_indexer(
    labeled_retrieval_eval.index
)

print("Human-labeled retrieval queries:", len(labeled_eval_positions))

# ---------------------------------------------------------
# 4. Evaluate Top-K intent consistency
# ---------------------------------------------------------

results = []

for k in [1, 3, 5, 10]:

    correct = 0
    evaluated = 0

    for eval_pos, eval_index in zip(
        labeled_eval_positions,
        labeled_retrieval_eval.index
    ):

        # True human intent
        query_intent = labeled_retrieval_eval.loc[
            eval_index,
            "human_intent"
        ]

        # Top-k retrieved examples
        top_k_indices = np.argsort(
            similarity_scores[eval_pos]
        )[::-1][:k]

        # Get intents of retrieved examples
        retrieved_intents = [
            retrieval_train_intents[i]
            for i in top_k_indices
            if retrieval_train_intents[i] is not None
        ]

        # Skip if retrieved examples have no human labels
        if not retrieved_intents:
            continue

        # Majority intent
        majority_intent = Counter(
            retrieved_intents
        ).most_common(1)[0][0]

        if majority_intent == query_intent:
            correct += 1

        evaluated += 1

    accuracy = (
        correct / evaluated
        if evaluated > 0
        else 0
    )

    results.append({
        "k": k,
        "correct": correct,
        "evaluated": evaluated,
        "intent_consistency": accuracy
    })


# ---------------------------------------------------------
# 5. Display results
# ---------------------------------------------------------

retrieval_intent_results = pd.DataFrame(results)

display(retrieval_intent_results)

Human-labeled retrieval queries: 28


,k,correct,evaluated,intent_consistency
0,1,5,5,1.000000
1,3,5,5,1.000000
2,5,7,10,0.700000
3,10,8,13,0.615385


In [92]:
# Cell 77 — Retrieval Quality Summary + Example Inspection

# ---------------------------------------------------------
# 1. Summary of semantic similarity
# ---------------------------------------------------------

retrieval_quality_summary = pd.DataFrame({
    "k": [1, 3, 5, 10],
    "mean_similarity": [
        similarity_scores.max(axis=1).mean(),
        np.sort(similarity_scores, axis=1)[:, -3:].mean(),
        np.sort(similarity_scores, axis=1)[:, -5:].mean(),
        np.sort(similarity_scores, axis=1)[:, -10:].mean()
    ]
})

display(retrieval_quality_summary)


# ---------------------------------------------------------
# 2. Inspect a few human-labeled retrieval queries
# ---------------------------------------------------------

print("Example retrieval results")
print("=" * 80)

for eval_pos, eval_index in zip(
    labeled_eval_positions[:5],
    labeled_retrieval_eval.index[:5]
):

    query = retrieval_eval.loc[
        eval_index,
        "customer_message"
    ]

    true_intent = labeled_retrieval_eval.loc[
        eval_index,
        "human_intent"
    ]

    top_indices = np.argsort(
        similarity_scores[eval_pos]
    )[::-1][:3]

    print("\nCUSTOMER QUERY:")
    print(query)

    print("\nHUMAN INTENT:")
    print(true_intent)

    print("\nTOP RETRIEVED EXAMPLES:")

    for rank, idx in enumerate(top_indices, start=1):

        print(f"\n{rank}. Similarity: {similarity_scores[eval_pos][idx]:.3f}")

        print(
            "Customer:",
            retrieval_train.iloc[idx]["customer_message"]
        )

        print(
            "Support:",
            retrieval_train.iloc[idx]["support_response"]
        )

    print("\n" + "-" * 80)

,k,mean_similarity
0,1,0.662866
1,3,0.629633
2,5,0.610707
3,10,0.581477


Example retrieval results

CUSTOMER QUERY:
I’ve tried trashing Bluetooth prefs - but this didn’t help. Could you please explain how to run the hardware test?

HUMAN INTENT:
connectivity

TOP RETRIEVED EXAMPLES:

1. Similarity: 0.469
Customer: Just upgraded from 6 to iPhone7; but Bluetooth connectivity is a major issue with iOS11. bt device automatically disconnects. Any help?
Support: @143871 Congratulations on the new iPhone 7! We want to be sure you can use your Bluetooth connection as you'd expect to. 
Let us know if the steps outlined here help you: https://t.co/JMbUWgME3J

2. Similarity: 0.443
Customer: the Bluetooth function on my iPhone is super frustrating! I accidentally pressed forget device and now my iPhone is forgetting my wireless headphone forever! I did all reset but still can’t find my device! so frustrating!
Support: @137170 We'd be happy to help you with this.  Let us know in DM the iOS version you're using and the type of Bluetooth headphones. https://t.co/GDrqU22Yp

In [93]:
# Cell 78 — Retrieval Intent Recall@K

results = []

for k in [1, 3, 5, 10]:

    hits = 0
    evaluated = 0

    for eval_pos, eval_index in zip(
        labeled_eval_positions,
        labeled_retrieval_eval.index
    ):

        query_intent = labeled_retrieval_eval.loc[
            eval_index,
            "human_intent"
        ]

        top_k_indices = np.argsort(
            similarity_scores[eval_pos]
        )[::-1][:k]

        retrieved_intents = [
            retrieval_train_intents[i]
            for i in top_k_indices
            if retrieval_train_intents[i] is not None
        ]

        if not retrieved_intents:
            continue

        evaluated += 1

        # Does at least one retrieved example
        # have the same intent as the query?
        if query_intent in retrieved_intents:
            hits += 1

    recall_at_k = (
        hits / evaluated
        if evaluated > 0
        else 0
    )

    results.append({
        "k": k,
        "hits": hits,
        "evaluated": evaluated,
        "intent_recall_at_k": recall_at_k
    })

retrieval_recall_results = pd.DataFrame(results)

display(retrieval_recall_results)

,k,hits,evaluated,intent_recall_at_k
0,1,5,5,1.000000
1,3,5,5,1.000000
2,5,7,10,0.700000
3,10,8,13,0.615385


In [94]:
# Cell 79 — Check Retrieval Conversation Leakage

leakage_count = 0
total_queries = len(retrieval_eval)

for eval_pos, eval_index in enumerate(retrieval_eval.index):

    query_conversation_id = retrieval_eval.loc[
        eval_index,
        "conversation_id"
    ]

    top_k_indices = np.argsort(
        similarity_scores[eval_pos]
    )[::-1][:10]

    retrieved_conversation_ids = retrieval_train.iloc[
        top_k_indices
    ]["conversation_id"].values

    if query_conversation_id in retrieved_conversation_ids:
        leakage_count += 1

print("Evaluation queries:", total_queries)
print("Queries with conversation leakage:", leakage_count)
print(
    "Leakage rate:",
    f"{leakage_count / total_queries:.2%}"
)

Evaluation queries: 622
Queries with conversation leakage: 50
Leakage rate: 8.04%


In [95]:
# Cell 80 — Conversation-Level Retrieval Split

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# 1. Get unique conversation IDs
# ---------------------------------------------------------

unique_conversations = (
    support_pairs["conversation_id"]
    .drop_duplicates()
    .tolist()
)

print(
    "Unique conversations:",
    len(unique_conversations)
)


# ---------------------------------------------------------
# 2. Split conversations, not individual messages
# ---------------------------------------------------------

train_conversations, eval_conversations = train_test_split(
    unique_conversations,
    test_size=0.20,
    random_state=42
)


# ---------------------------------------------------------
# 3. Build train/evaluation datasets
# ---------------------------------------------------------

retrieval_train_v2 = support_pairs[
    support_pairs["conversation_id"].isin(train_conversations)
].copy()

retrieval_eval_v2 = support_pairs[
    support_pairs["conversation_id"].isin(eval_conversations)
].copy()


# ---------------------------------------------------------
# 4. Verify conversation overlap
# ---------------------------------------------------------

train_conversation_set = set(
    retrieval_train_v2["conversation_id"]
)

eval_conversation_set = set(
    retrieval_eval_v2["conversation_id"]
)

conversation_overlap = (
    train_conversation_set &
    eval_conversation_set
)


# ---------------------------------------------------------
# 5. Verify exact customer-message overlap
# ---------------------------------------------------------

train_messages = set(
    retrieval_train_v2["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

eval_messages = set(
    retrieval_eval_v2["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

message_overlap = train_messages & eval_messages


# ---------------------------------------------------------
# 6. Print results
# ---------------------------------------------------------

print("\nRetrieval train rows:", len(retrieval_train_v2))
print("Retrieval evaluation rows:", len(retrieval_eval_v2))

print(
    "\nTrain conversations:",
    len(train_conversation_set)
)

print(
    "Evaluation conversations:",
    len(eval_conversation_set)
)

print(
    "\nConversation overlap:",
    len(conversation_overlap)
)

print(
    "Exact customer-message overlap:",
    len(message_overlap)
)

Unique conversations: 2066

Retrieval train rows: 2475
Retrieval evaluation rows: 617

Train conversations: 1652
Evaluation conversations: 414

Conversation overlap: 0
Exact customer-message overlap: 9


In [96]:
# Cell 81 — Remove Exact Message Overlap

# Normalize customer messages
retrieval_train_v2["normalized_text"] = (
    retrieval_train_v2["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

retrieval_eval_v2["normalized_text"] = (
    retrieval_eval_v2["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Find messages that appear in training data
train_message_set = set(
    retrieval_train_v2["normalized_text"]
)

# Identify overlapping evaluation rows
eval_overlap_mask = (
    retrieval_eval_v2["normalized_text"]
    .isin(train_message_set)
)

overlap_count = eval_overlap_mask.sum()

print(
    "Evaluation rows with exact message overlap:",
    overlap_count
)

# Remove overlapping evaluation examples
retrieval_eval_final = retrieval_eval_v2[
    ~eval_overlap_mask
].copy()

# Remove helper column
retrieval_train_v2 = retrieval_train_v2.drop(
    columns=["normalized_text"]
)

retrieval_eval_final = retrieval_eval_final.drop(
    columns=["normalized_text"]
)

print(
    "\nFinal retrieval train rows:",
    len(retrieval_train_v2)
)

print(
    "Final retrieval evaluation rows:",
    len(retrieval_eval_final)
)

# Final verification
train_messages_final = set(
    retrieval_train_v2["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

eval_messages_final = set(
    retrieval_eval_final["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

remaining_overlap = (
    train_messages_final &
    eval_messages_final
)

train_conversations_final = set(
    retrieval_train_v2["conversation_id"]
)

eval_conversations_final = set(
    retrieval_eval_final["conversation_id"]
)

conversation_overlap_final = (
    train_conversations_final &
    eval_conversations_final
)

print(
    "\nRemaining exact message overlap:",
    len(remaining_overlap)
)

print(
    "Remaining conversation overlap:",
    len(conversation_overlap_final)
)

Evaluation rows with exact message overlap: 14

Final retrieval train rows: 2475
Final retrieval evaluation rows: 603

Remaining exact message overlap: 0
Remaining conversation overlap: 0


In [97]:
# Cell 82 — Fresh Leakage-Safe Retrieval Embeddings

from sentence_transformers import SentenceTransformer

# Load embedding model
retrieval_embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Prepare texts
retrieval_train_texts_final = (
    retrieval_train_v2["customer_message"]
    .astype(str)
    .str.strip()
    .tolist()
)

retrieval_eval_texts_final = (
    retrieval_eval_final["customer_message"]
    .astype(str)
    .str.strip()
    .tolist()
)

print("Encoding retrieval training data...")

retrieval_train_embeddings_final = (
    retrieval_embedding_model.encode(
        retrieval_train_texts_final,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

print("\nEncoding retrieval evaluation data...")

retrieval_eval_embeddings_final = (
    retrieval_embedding_model.encode(
        retrieval_eval_texts_final,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

print("\nFinal embedding shapes:")
print(
    "Train:",
    retrieval_train_embeddings_final.shape
)

print(
    "Evaluation:",
    retrieval_eval_embeddings_final.shape
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 714.26it/s]


Encoding retrieval training data...


Batches: 100%|██████████| 78/78 [00:45<00:00,  1.73it/s]



Encoding retrieval evaluation data...


Batches: 100%|██████████| 19/19 [00:12<00:00,  1.54it/s]


Final embedding shapes:
Train: (2475, 384)
Evaluation: (603, 384)


In [98]:
# Cell 83 — Final Leakage-Safe Retrieval Similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Calculate cosine similarity between every evaluation query
# and every training customer message
final_similarity_scores = cosine_similarity(
    retrieval_eval_embeddings_final,
    retrieval_train_embeddings_final
)

print("Similarity matrix shape:")
print(final_similarity_scores.shape)

# Evaluate different Top-K values
top_k_values = [1, 3, 5, 10]

final_retrieval_results = []

for k in top_k_values:

    # Get top-k similarities for every evaluation query
    top_k_similarities = np.sort(
        final_similarity_scores,
        axis=1
    )[:, -k:]

    final_retrieval_results.append({
        "k": k,
        "mean_similarity": top_k_similarities.mean(),
        "median_similarity": np.median(top_k_similarities),
        "min_similarity": top_k_similarities.min(),
        "max_similarity": top_k_similarities.max()
    })

final_retrieval_results = pd.DataFrame(
    final_retrieval_results
)

display(final_retrieval_results)

Similarity matrix shape:
(603, 2475)


,k,mean_similarity,median_similarity,min_similarity,max_similarity
0,1,0.648590,0.646206,0.293191,0.943867
1,3,0.614394,0.609960,0.203194,0.943867
2,5,0.595317,0.589161,0.202053,0.943867
3,10,0.567070,0.561016,0.179005,0.943867


In [99]:
# Cell 84 — Human-Labeled Examples in Final Retrieval Evaluation

# Build lookup from human-validated data
human_intent_map_final = {
    str(row["text"]).lower().strip(): row["intent"]
    for _, row in human_validated_df.iterrows()
}

# Normalize evaluation messages
retrieval_eval_final["normalized_text"] = (
    retrieval_eval_final["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Attach human intent where available
retrieval_eval_final["human_intent"] = (
    retrieval_eval_final["normalized_text"]
    .map(human_intent_map_final)
)

# Keep only human-labeled examples
final_labeled_retrieval_eval = retrieval_eval_final[
    retrieval_eval_final["human_intent"].notna()
].copy()

print(
    "Human-labeled examples in final retrieval evaluation:",
    len(final_labeled_retrieval_eval)
)

print("\nIntent distribution:")
display(
    final_labeled_retrieval_eval["human_intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

Human-labeled examples in final retrieval evaluation: 15

Intent distribution:


,intent,count
0,other_unclear,8
1,keyboard_input,2
2,battery_power,2
3,device_hardware,1
4,software_update,1
5,apps_app_store,1


In [100]:
# Cell 85 — Final Leakage-Safe Intent Recall@K

from collections import Counter
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Build human intent lookup for retrieval training data
# ---------------------------------------------------------

retrieval_train_intents_final = []

for _, row in retrieval_train_v2.iterrows():

    text = str(row["customer_message"]).lower().strip()

    intent = human_intent_map_final.get(text, None)

    retrieval_train_intents_final.append(intent)

retrieval_train_intents_final = np.array(
    retrieval_train_intents_final,
    dtype=object
)


# ---------------------------------------------------------
# 2. Get positions of human-labeled evaluation examples
# ---------------------------------------------------------

final_labeled_positions = (
    retrieval_eval_final.index.get_indexer(
        final_labeled_retrieval_eval.index
    )
)

print(
    "Human-labeled evaluation queries:",
    len(final_labeled_positions)
)


# ---------------------------------------------------------
# 3. Evaluate Recall@K
# ---------------------------------------------------------

results = []

for k in [1, 3, 5, 10]:

    hits = 0
    evaluated = 0

    for eval_pos, eval_index in zip(
        final_labeled_positions,
        final_labeled_retrieval_eval.index
    ):

        query_intent = final_labeled_retrieval_eval.loc[
            eval_index,
            "human_intent"
        ]

        # Top-k retrieved examples
        top_k_indices = np.argsort(
            final_similarity_scores[eval_pos]
        )[::-1][:k]

        # Human intents available for retrieved examples
        retrieved_intents = [
            retrieval_train_intents_final[i]
            for i in top_k_indices
            if retrieval_train_intents_final[i] is not None
        ]

        if not retrieved_intents:
            continue

        evaluated += 1

        # Recall@K:
        # Did at least one retrieved example
        # have the same human intent?
        if query_intent in retrieved_intents:
            hits += 1

    recall = (
        hits / evaluated
        if evaluated > 0
        else 0
    )

    results.append({
        "k": k,
        "hits": hits,
        "evaluated": evaluated,
        "intent_recall_at_k": recall
    })


final_retrieval_recall = pd.DataFrame(results)

display(final_retrieval_recall)

Human-labeled evaluation queries: 15


,k,hits,evaluated,intent_recall_at_k
0,1,0,0,0.000000
1,3,0,1,0.000000
2,5,0,2,0.000000
3,10,1,3,0.333333


In [101]:
# Cell 86 — Dedicated Human-Validated Retrieval Evaluation

# ---------------------------------------------------------
# 1. Normalize human validation messages
# ---------------------------------------------------------

human_eval = human_validated_df.copy()

human_eval["normalized_text"] = (
    human_eval["text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# ---------------------------------------------------------
# 2. Find conversations containing human-evaluated messages
# ---------------------------------------------------------

human_eval_conversations = set()

for text in human_eval["normalized_text"]:

    matching_rows = support_pairs[
        support_pairs["customer_message"]
        .astype(str)
        .str.lower()
        .str.strip()
        == text
    ]

    human_eval_conversations.update(
        matching_rows["conversation_id"].tolist()
    )

print(
    "Human evaluation examples:",
    len(human_eval)
)

print(
    "Conversations removed from retrieval corpus:",
    len(human_eval_conversations)
)


# ---------------------------------------------------------
# 3. Build leakage-safe retrieval corpus
# ---------------------------------------------------------

dedicated_retrieval_corpus = support_pairs[
    ~support_pairs["conversation_id"].isin(
        human_eval_conversations
    )
].copy()

print(
    "\nDedicated retrieval corpus rows:",
    len(dedicated_retrieval_corpus)
)


# ---------------------------------------------------------
# 4. Check exact message overlap
# ---------------------------------------------------------

human_eval_texts = set(
    human_eval["normalized_text"]
)

corpus_texts = set(
    dedicated_retrieval_corpus["customer_message"]
    .astype(str)
    .str.lower()
    .str.strip()
)

message_overlap = (
    human_eval_texts &
    corpus_texts
)

print(
    "Exact message overlap:",
    len(message_overlap)
)


# ---------------------------------------------------------
# 5. Check conversation overlap
# ---------------------------------------------------------

remaining_conversations = set(
    dedicated_retrieval_corpus["conversation_id"]
)

conversation_overlap = (
    human_eval_conversations &
    remaining_conversations
)

print(
    "Conversation overlap:",
    len(conversation_overlap)
)

Human evaluation examples: 120
Conversations removed from retrieval corpus: 145

Dedicated retrieval corpus rows: 2791
Exact message overlap: 0
Conversation overlap: 0


In [102]:
# Cell 87 — Dedicated Human-Validated Retrieval Embeddings

from sentence_transformers import SentenceTransformer

# Load the same embedding model
dedicated_embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# ---------------------------------------------------------
# 1. Prepare retrieval corpus
# ---------------------------------------------------------

dedicated_retrieval_texts = (
    dedicated_retrieval_corpus["customer_message"]
    .astype(str)
    .str.strip()
    .tolist()
)

# ---------------------------------------------------------
# 2. Prepare all 120 human evaluation queries
# ---------------------------------------------------------

human_eval_texts = (
    human_eval["text"]
    .astype(str)
    .str.strip()
    .tolist()
)

# ---------------------------------------------------------
# 3. Encode retrieval corpus
# ---------------------------------------------------------

print("Encoding retrieval corpus...")

dedicated_retrieval_embeddings = (
    dedicated_embedding_model.encode(
        dedicated_retrieval_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

# ---------------------------------------------------------
# 4. Encode human evaluation queries
# ---------------------------------------------------------

print("\nEncoding human evaluation queries...")

human_eval_embeddings = (
    dedicated_embedding_model.encode(
        human_eval_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

# ---------------------------------------------------------
# 5. Verify shapes
# ---------------------------------------------------------

print("\nEmbedding shapes:")

print(
    "Retrieval corpus:",
    dedicated_retrieval_embeddings.shape
)

print(
    "Human evaluation:",
    human_eval_embeddings.shape
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 550.55it/s]


Encoding retrieval corpus...


Batches: 100%|██████████| 88/88 [00:55<00:00,  1.57it/s]



Encoding human evaluation queries...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.72it/s]



Embedding shapes:
Retrieval corpus: (2791, 384)
Human evaluation: (120, 384)


In [103]:
# Cell 88 — Final Retrieval Intent Recall@K

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Calculate similarity between all 120 queries
#    and the leakage-safe retrieval corpus
# ---------------------------------------------------------

dedicated_similarity_scores = cosine_similarity(
    human_eval_embeddings,
    dedicated_retrieval_embeddings
)

print(
    "Similarity matrix shape:",
    dedicated_similarity_scores.shape
)


# ---------------------------------------------------------
# 2. Build intent lookup for retrieval corpus
# ---------------------------------------------------------

human_intent_lookup = {
    str(row["text"]).lower().strip(): row["intent"]
    for _, row in human_validated_df.iterrows()
}

dedicated_retrieval_intents = []

for _, row in dedicated_retrieval_corpus.iterrows():

    text = (
        str(row["customer_message"])
        .lower()
        .strip()
    )

    dedicated_retrieval_intents.append(
        human_intent_lookup.get(text, None)
    )

dedicated_retrieval_intents = np.array(
    dedicated_retrieval_intents,
    dtype=object
)


# ---------------------------------------------------------
# 3. Evaluate Intent Recall@K
# ---------------------------------------------------------

results = []

for k in [1, 3, 5, 10]:

    hits = 0
    evaluated = 0

    for query_idx in range(len(human_eval)):

        query_intent = human_eval.iloc[
            query_idx
        ]["intent"]

        top_k_indices = np.argsort(
            dedicated_similarity_scores[query_idx]
        )[::-1][:k]

        retrieved_intents = [
            dedicated_retrieval_intents[i]
            for i in top_k_indices
            if dedicated_retrieval_intents[i] is not None
        ]

        # Only count queries where retrieved examples
        # have human labels available.
        if not retrieved_intents:
            continue

        evaluated += 1

        if query_intent in retrieved_intents:
            hits += 1

    recall = (
        hits / evaluated
        if evaluated > 0
        else 0
    )

    results.append({
        "k": k,
        "hits": hits,
        "evaluated": evaluated,
        "intent_recall_at_k": recall
    })


final_intent_recall = pd.DataFrame(results)

display(final_intent_recall)

Similarity matrix shape: (120, 2791)


,k,hits,evaluated,intent_recall_at_k
0,1,0,0,0
1,3,0,0,0
2,5,0,0,0
3,10,0,0,0


In [104]:
# Cell 89 — Generate Manual Retrieval Review Table

manual_review_rows = []

# Review all 120 human-validated queries
for query_idx in range(len(human_eval)):

    query_text = human_eval.iloc[query_idx]["text"]
    true_intent = human_eval.iloc[query_idx]["intent"]

    # Top 3 retrieved examples
    top_indices = np.argsort(
        dedicated_similarity_scores[query_idx]
    )[::-1][:3]

    for rank, idx in enumerate(top_indices, start=1):

        manual_review_rows.append({
            "query_id": query_idx + 1,
            "query": query_text,
            "human_intent": true_intent,
            "rank": rank,
            "similarity": round(
                float(
                    dedicated_similarity_scores[
                        query_idx
                    ][idx]
                ),
                3
            ),
            "retrieved_customer": (
                dedicated_retrieval_corpus.iloc[idx][
                    "customer_message"
                ]
            ),
            "retrieved_support": (
                dedicated_retrieval_corpus.iloc[idx][
                    "support_response"
                ]
            )
        })

manual_review_df = pd.DataFrame(
    manual_review_rows
)

print(
    "Total retrieval examples:",
    len(manual_review_df)
)

display(
    manual_review_df.head(15)
)

Total retrieval examples: 360


,query_id,query,human_intent,rank,similarity,retrieved_customer,retrieved_support
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,1,0.685,iOS11 I’ve tried resetting the keyboard it didn’t work.,@122931 Join us in DM and send us a screenshot of Settings &gt; General &gt; About. https://t.co/GDrqU22YpT
1,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,2,0.632,"Dear @115858 , I’m tired of not being to press send on my quick replies because the keyboard covers it! #IOS11 #PleaseFixIt https://t.co/T2NfcZIPNj",@129788 We're here to help. Let's take this to DM so we can better assist you. https://t.co/GDrqU22YpT
2,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,3,0.610,"Hey and anyone else who upgraded to ios11.1, are y’all having issues with capital “I” in the Mail app? As it puts in “A”?","@115856 Hey, let's work together to figure out what's going on. Meet us in DM and we'll continue from there. https://t.co/GDrqU22YpT"
3,2,I am using an iPhone 8 Plus and iOS 11.1.2,other_unclear,1,0.941,I am using Iphone 7 and iOS 11.1,"@123515 DM us your country, and we'll continue working there. https://t.co/GDrqU22YpT"
4,2,I am using an iPhone 8 Plus and iOS 11.1.2,other_unclear,2,0.880,i am using ios 11.1.2,@142178 Thanks. Do you only experience issues with AirDrop when attempting to share with your MacBook?
5,2,I am using an iPhone 8 Plus and iOS 11.1.2,other_unclear,3,0.873,I have an iPhone 7 running iOS 11.1.2,@138689 This article provides the steps for how to adjust the brightness on your iPhone 7: https://t.co/QGzyuyr5Ra Let us know if this helps.
6,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",performance_freezing,1,0.692,"iPhone 7+, freezes all the time, need to hard reset at least twice a day since iOS 11 update","@125603 Thank you for those details, and we'll be happy to help. Send us a DM so we can look further into this issue. https://t.co/GDrqU22YpT"
7,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",performance_freezing,2,0.648,"iPhone 7 and I updated to the newest ios but the phone started crashing and the screen freezing, also powering off",@123686 We'd like to help out with this. Join us in DM so that we can gather more details about your device. https://t.co/GDrqU22YpT
8,3,"pls help,my ip7+ freeze all the time after update ios 11.1.2 & the camera is also got blurry,also couldnt open my photo gallery.what happen?",performance_freezing,3,0.647,"iPhone 7 Plus camera doesn't focus anymore when in Photo, Square mode since upgrading to iOS 11.",@123526 We can help with this. Are you on the latest iOS 11.1? DM us your reply to continue https://t.co/GDrqU22YpT
9,4,Updated to iOS 11.1.2 and now i have no sound 👍🏼 @115858 #sickupdatedudes,software_update,1,0.721,This is the sound iOS 11.1.2,@121004 Thanks for that video. Let's look deeper into this with you. DM us what country you are in and we'll continue from there. https://t.co/GDrqU22YpT


In [105]:
# Cell 90 — Manual Retrieval Relevance Review

# Select 30 queries evenly across the 120 human-validated examples
review_indices = np.linspace(
    0,
    len(human_eval) - 1,
    30,
    dtype=int
)

manual_review = []

for query_idx in review_indices:

    query = human_eval.iloc[query_idx]

    # Top-1 retrieval
    top_idx = np.argmax(
        dedicated_similarity_scores[query_idx]
    )

    manual_review.append({
        "review_id": len(manual_review) + 1,
        "query": query["text"],
        "human_intent": query["intent"],
        "similarity": round(
            float(
                dedicated_similarity_scores[
                    query_idx
                ][top_idx]
            ),
            3
        ),
        "retrieved_customer": (
            dedicated_retrieval_corpus.iloc[top_idx][
                "customer_message"
            ]
        ),
        "retrieved_support": (
            dedicated_retrieval_corpus.iloc[top_idx][
                "support_response"
            ]
        ),
        "relevance": ""
    })

manual_review_df = pd.DataFrame(
    manual_review
)

print(
    "Manual review examples:",
    len(manual_review_df)
)

display(manual_review_df)

Manual review examples: 30


,review_id,query,human_intent,similarity,retrieved_customer,retrieved_support,relevance
0,1,Is it just me or did the delete key on my Bluetooth keyboard stop working in Mail on iOS11? Com’on @115858 you’re killing me with iOS11 bugs. https://t.co/QpvkrEvzIg,keyboard_input,0.685,iOS11 I’ve tried resetting the keyboard it didn’t work.,@122931 Join us in DM and send us a screenshot of Settings &gt; General &gt; About. https://t.co/GDrqU22YpT,
1,2,"About 2 weeks, I have IOS 11.1.2",other_unclear,0.743,When the update will be available? iOS 11.1 .,@123833 iOS 11.1 is available now. You can update using the steps here: https://t.co/80YRnjDFDk,
2,3,ios 11,other_unclear,0.899,iOS 11.1,"@140196 We recently released an iOS update, 11.1.2. Let’s make sure there are no available updates on your device. Be sure to back up before doing an update. How to back up: https://t.co/tHicJHB57O",
3,4,"I promise that I did the workouts to finish the Thanksgiving day challenge. Yet, no badge. Does have any ideas?",other_unclear,0.696,completed a run today and my Thanksgiving badge hasn’t shown up :( Any advice?,@142182 Let's have a closer look. Please DM us. https://t.co/GDrqU22YpT,
4,5,@AppleSupport https://t.co/yvu10fymce,other_unclear,0.942,@AppleSupport https://t.co/NV0yucs0lB,@115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.,
5,6,11.0.2,other_unclear,0.952,11.0.1,@127929 We see that an update is available and recommend installing it. Check out the following article: https://t.co/80YRnjDFDk,
6,7,11.1,other_unclear,0.941,11.0.1,@127929 We see that an update is available and recommend installing it. Check out the following article: https://t.co/80YRnjDFDk,
7,8,I have 11.0.3,other_unclear,0.867,11.0.3,"@138163 Let's backup your device, then update to the latest version of iOS, as shown here: https://t.co/80YRnjDFDk Then test it out and let us know what you see. Shoot us a DM if the issue persists by tapping here: https://t.co/GDrqU22YpT",
8,9,Man I’m having the same problem. Thought EYE was alone. Was just about to accept life without ever using the word “I” anymore.,keyboard_input,0.707,every time “eye” type “eye” it’s turns to I... HELP!,@127768 We'll be happy to assist. When did this start to happen? Are you running iOS 11.1 yet?,
9,10,For those that have been wondering. why are y’all punishing us for using pronouns!?,keyboard_input,0.541,WHY IS MY AUTOCORRECT CORRECTING “it” to “I.T” all of a sudden😡😡 it’s just creepy and very annoying honestly. @115858,"@134509 We totally get that, but no worries, we can help! Let's start by having you tell us which iOS version is running under Settings &gt; General &gt; About. We'll get started there.",


In [108]:
# Cell 91 — Calculate Manual Retrieval Relevance

manual_review_df["relevance"] = [
    "relevant",
    "partial",
    "relevant",
    "relevant",
    "partial",
    "partial",
    "partial",
    "relevant",
    "relevant",
    "relevant",
    "relevant",
    "partial",
    "partial",
    "irrelevant",
    "partial",
    "relevant",
    "partial",
    "partial",
    "relevant",
    "relevant",
    "irrelevant",
    "partial",
    "relevant",
    "relevant",
    "relevant",
    "relevant",
    "partial",
    "partial",
    "partial",
    "partial"
]

print("Manual retrieval evaluation")
print("=" * 40)

counts = manual_review_df["relevance"].value_counts()

display(counts)

total = len(manual_review_df)

relevant_rate = (
    (manual_review_df["relevance"] == "relevant").sum()
    / total
)

relevant_or_partial_rate = (
    manual_review_df["relevance"]
    .isin(["relevant", "partial"])
    .sum()
    / total
)

irrelevant_rate = (
    (manual_review_df["relevance"] == "irrelevant").sum()
    / total
)

print(f"Directly relevant: {relevant_rate:.1%}")
print(f"Relevant or partial: {relevant_or_partial_rate:.1%}")
print(f"Irrelevant: {irrelevant_rate:.1%}")

Manual retrieval evaluation


relevance
relevant      14
partial       14
irrelevant     2
Name: count, dtype: int64

Directly relevant: 46.7%
Relevant or partial: 93.3%
Irrelevant: 6.7%


In [109]:
# Cell 92 — End-to-End Agent Decision Policy

import numpy as np

CONFIDENCE_THRESHOLD = 0.20


def decide_agent_action(
    intent_confidence,
    retrieval_similarity,
    retrieval_relevance=None
):
    """
    Decide what the support agent should do.

    Policy:
    1. Low intent confidence -> clarify / escalate
    2. Strong intent confidence + weak retrieval -> clarify / escalate
    3. Strong intent confidence + strong retrieval -> use retrieved evidence
    """

    # Step 1: Intent confidence check
    if intent_confidence < CONFIDENCE_THRESHOLD:
        return "clarify_or_escalate"

    # Step 2: Retrieval evidence check
    #
    # We use a conservative similarity threshold.
    # This is a routing heuristic, not a learned ground truth.
    if retrieval_similarity < 0.60:
        return "clarify_or_escalate"

    # Step 3: If manual relevance is available,
    # do not trust an explicitly irrelevant result.
    if retrieval_relevance == "irrelevant":
        return "clarify_or_escalate"

    return "use_retrieved_evidence"


print("Agent decision policy created.")
print()
print("Policy:")
print("1. Low intent confidence → clarify / escalate")
print("2. Weak retrieval evidence → clarify / escalate")
print("3. Strong intent + retrieval → use retrieved evidence")

Agent decision policy created.

Policy:
1. Low intent confidence → clarify / escalate
2. Weak retrieval evidence → clarify / escalate
3. Strong intent + retrieval → use retrieved evidence


In [110]:
# Cell 93 — End-to-End Agent Function

import numpy as np


def run_support_agent(customer_message):
    """
    Run the complete support pipeline:

    Customer message
        ↓
    Intent classification
        ↓
    Semantic retrieval
        ↓
    Decision policy
    """

    # -----------------------------
    # 1. Intent classification
    # -----------------------------
    query_embedding = embedding_model.encode(
        [customer_message],
        normalize_embeddings=True
    )

    intent_probabilities = final_intent_classifier.predict_proba(
        query_embedding
    )[0]

    predicted_index = np.argmax(intent_probabilities)
    predicted_intent = final_intent_classifier.classes_[predicted_index]
    intent_confidence = float(
        intent_probabilities[predicted_index]
    )

    # -----------------------------
    # 2. Retrieval
    # -----------------------------
    query_embedding = np.asarray(query_embedding)

    similarity_scores = (
        query_embedding @ dedicated_retrieval_embeddings.T
    )[0]

    top_index = np.argmax(similarity_scores)

    retrieved_row = dedicated_retrieval_corpus.iloc[top_index]

    retrieval_similarity = float(
        similarity_scores[top_index]
    )

    # -----------------------------
    # 3. Agent decision
    # -----------------------------
    decision = decide_agent_action(
        intent_confidence=intent_confidence,
        retrieval_similarity=retrieval_similarity
    )

    # -----------------------------
    # 4. Return complete result
    # -----------------------------
    return {
        "customer_message": customer_message,
        "predicted_intent": predicted_intent,
        "intent_confidence": round(intent_confidence, 3),
        "retrieved_customer": retrieved_row["customer_message"],
        "retrieved_support": retrieved_row["support_response"],
        "retrieval_similarity": round(retrieval_similarity, 3),
        "decision": decision
    }


print("End-to-end support agent created.")

End-to-end support agent created.


In [113]:
# Cell 94 — Test End-to-End Agent

test_messages = [
    "My battery is draining very quickly after updating to iOS 11.",
    "My keyboard keeps changing the word I to something else.",
    "I cannot log in to my Apple ID.",
    "My iPhone keeps freezing after the latest update.",
    "I have a problem with my phone."
]

for i, message in enumerate(test_messages, 1):

    result = run_support_agent(message)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {result['customer_message']}")
    print(f"Predicted intent: {result['predicted_intent']}")
    print(f"Intent confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nRetrieved customer:")
    print(result["retrieved_customer"])

    print("\nRetrieved support:")
    print(result["retrieved_support"])

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Predicted intent: battery_power
Intent confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Retrieved customer:
since I’ve updated to the IOS 11 my battery has been draining to quickly I’ve only been using it for 15 mins& it’s drained

Retrieved support:
@134147 That's unexpected, but we'd be happy to help. Let's move over to DM. Let us know which iOS version you currently have installed. You can check in Settings &gt; General &gt; About. https://t.co/GDrqU22YpT
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Predicted intent: keyboard_input
Intent confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Retrieved customer:
how do I get my keyboard to STOP auto correcting the letter "I" to a symbol. See attached photo.

Retrieved support:
@128907 We want to look into this with you. DM us and we'll continue there. https://t.co/GDrqU22

In [114]:
# Cell 95 — Structured End-to-End Evaluation

evaluation_messages = [
    "My battery is draining very quickly after updating to iOS 11.",
    "My keyboard keeps changing the word I to something else.",
    "I cannot log in to my Apple ID.",
    "My iPhone keeps freezing after the latest update.",
    "I have a problem with my phone."
]

evaluation_results = []

for message in evaluation_messages:

    result = run_support_agent(message)

    evaluation_results.append({
        "customer_message": result["customer_message"],
        "predicted_intent": result["predicted_intent"],
        "intent_confidence": result["intent_confidence"],
        "retrieval_similarity": result["retrieval_similarity"],
        "decision": result["decision"]
    })

end_to_end_df = pd.DataFrame(evaluation_results)

display(end_to_end_df)

print("\nDecision distribution:")
display(
    end_to_end_df["decision"].value_counts()
)

,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision
0,My battery is draining very quickly after updating to iOS 11.,battery_power,0.404,0.882,use_retrieved_evidence
1,My keyboard keeps changing the word I to something else.,keyboard_input,0.514,0.825,use_retrieved_evidence
2,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate
3,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,use_retrieved_evidence
4,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate



Decision distribution:


decision
use_retrieved_evidence    3
clarify_or_escalate       2
Name: count, dtype: int64

In [115]:
# Cell 95 — Structured End-to-End Evaluation

evaluation_messages = [
    "My battery is draining very quickly after updating to iOS 11.",
    "My keyboard keeps changing the word I to something else.",
    "I cannot log in to my Apple ID.",
    "My iPhone keeps freezing after the latest update.",
    "I have a problem with my phone."
]

evaluation_results = []

for message in evaluation_messages:

    result = run_support_agent(message)

    evaluation_results.append({
        "customer_message": result["customer_message"],
        "predicted_intent": result["predicted_intent"],
        "intent_confidence": result["intent_confidence"],
        "retrieval_similarity": result["retrieval_similarity"],
        "decision": result["decision"]
    })

end_to_end_df = pd.DataFrame(evaluation_results)

display(end_to_end_df)

print("\nDecision distribution:")
display(
    end_to_end_df["decision"].value_counts()
)

,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision
0,My battery is draining very quickly after updating to iOS 11.,battery_power,0.404,0.882,use_retrieved_evidence
1,My keyboard keeps changing the word I to something else.,keyboard_input,0.514,0.825,use_retrieved_evidence
2,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate
3,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,use_retrieved_evidence
4,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate



Decision distribution:


decision
use_retrieved_evidence    3
clarify_or_escalate       2
Name: count, dtype: int64

In [116]:
# Cell 97 — Safe Response Generator

def generate_support_response(agent_result):
    """
    Generate a safe customer-facing response
    based on the agent decision.
    """

    decision = agent_result["decision"]

    if decision == "clarify_or_escalate":

        return (
            "I’d be happy to help. Could you share a few more "
            "details about the issue, such as your device model "
            "and the iOS version you are currently using?"
        )

    # If retrieval evidence is considered useful,
    # use the retrieved support guidance as evidence.
    retrieved_support = agent_result["retrieved_support"]

    if not retrieved_support:
        return (
            "I’d be happy to help. Please share more details "
            "about the issue so we can look into it."
        )

    return (
        "Thanks for reaching out. "
        + retrieved_support
    )


print("Safe response generator created.")

Safe response generator created.


In [117]:
# Cell 98 — Test Final Response Generation

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    response = generate_support_response(result)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Decision: {result['decision']}")
    print("\nSuggested response:")
    print(response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Decision: use_retrieved_evidence

Suggested response:
Thanks for reaching out. @134147 That's unexpected, but we'd be happy to help. Let's move over to DM. Let us know which iOS version you currently have installed. You can check in Settings &gt; General &gt; About. https://t.co/GDrqU22YpT
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Decision: use_retrieved_evidence

Suggested response:
Thanks for reaching out. @128907 We want to look into this with you. DM us and we'll continue there. https://t.co/GDrqU22YpT
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Decision: clarify_or_escalate

Suggested response:
I’d be happy to help. Could you share a few more details about the issue, such as your device model and the iOS version you are currently using?
TEST 4
Cu

In [118]:
# Cell 99 — Clean Retrieved Support Response

import re


def clean_support_response(response):
    """
    Remove Twitter-specific artifacts from historical support replies.
    Keep the useful support guidance.
    """

    text = str(response)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+", "", text)

    # Decode common HTML entities
    text = text.replace("&amp;", "&")
    text = text.replace("&gt;", ">")
    text = text.replace("&lt;", "<")

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test the cleaner on one retrieved response
example_response = end_to_end_df.iloc[0]

# Retrieve the original support response again
test_result = run_support_agent(
    example_response["customer_message"]
)

cleaned = clean_support_response(
    test_result["retrieved_support"]
)

print("Original:")
print(test_result["retrieved_support"])

print("\nCleaned:")
print(cleaned)

Original:
@134147 That's unexpected, but we'd be happy to help. Let's move over to DM. Let us know which iOS version you currently have installed. You can check in Settings &gt; General &gt; About. https://t.co/GDrqU22YpT

Cleaned:
That's unexpected, but we'd be happy to help. Let's move over to DM. Let us know which iOS version you currently have installed. You can check in Settings > General > About.


In [119]:
# Cell 100 — Improved Response Generator

def generate_support_response(agent_result):
    """
    Generate a cleaner customer-facing response
    from retrieved support evidence.
    """

    decision = agent_result["decision"]

    # Low-confidence cases
    if decision == "clarify_or_escalate":
        return (
            "I'd be happy to help. Could you share a few more "
            "details about the issue, including your device model "
            "and the iOS version you're currently using?"
        )

    # Clean historical support evidence
    evidence = clean_support_response(
        agent_result["retrieved_support"]
    )

    if not evidence:
        return (
            "I'd be happy to help. Could you share more details "
            "about the issue so we can look into it?"
        )

    # Remove generic Twitter-specific escalation wording
    evidence = re.sub(
        r"Let's move over to DM\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"DM us and we'll continue there\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"Please DM us\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"Send us a DM\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(r"\s+", " ", evidence).strip()

    return (
        "Thanks for reaching out. "
        + evidence
    )


print("Improved response generator created.")

Improved response generator created.


In [120]:
# Cell 101 — Test Improved Response Generator

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    response = generate_support_response(result)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. We want to look into this with you.
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Decision: clarify_or_escalate

Final suggested response:
I'd be happy to help. Could you share a few more details about the issue, including your device model and the iOS version you're currently using?
TEST 4
Customer: My iPhone keeps freezing after the latest update.
Intent: performance_freezing
Confidence: 0.249
D

In [121]:
# Cell 102 — Final Safer Response Generator

def generate_support_response(agent_result):
    """
    Generate a customer-facing response using retrieved evidence.

    The generator avoids copying Twitter-specific wording and
    avoids inventing unsupported troubleshooting instructions.
    """

    decision = agent_result["decision"]
    intent = agent_result["predicted_intent"]

    # -----------------------------------
    # 1. Low-confidence case
    # -----------------------------------
    if decision == "clarify_or_escalate":

        return (
            "I'd be happy to help. Could you share a few more "
            "details about the issue, including your device model "
            "and the iOS version you're currently using?"
        )

    # -----------------------------------
    # 2. Clean retrieved evidence
    # -----------------------------------
    evidence = clean_support_response(
        agent_result["retrieved_support"]
    )

    # -----------------------------------
    # 3. Remove Twitter-specific wording
    # -----------------------------------
    evidence = re.sub(
        r"Let's move over to DM\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"DM us and we'll continue there\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"Please DM us\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"Send us a DM\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"Let us know in DM,?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"we'll work with you there\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"We want to look into this with you\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"We want to help in any way we can\.?",
        "",
        evidence,
        flags=re.IGNORECASE
    )

    evidence = re.sub(
        r"\s+",
        " ",
        evidence
    ).strip()

    # -----------------------------------
    # 4. If useful evidence remains,
    #    return it.
    # -----------------------------------
    if evidence:
        return (
            "Thanks for reaching out. "
            + evidence
        )

    # -----------------------------------
    # 5. No useful retrieved guidance
    #    → ask intent-specific question
    # -----------------------------------
    clarification_questions = {
        "battery_power": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when you first noticed the battery drain?"
        ),

        "keyboard_input": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and which keyboard or typing issue you're seeing?"
        ),

        "performance_freezing": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when the freezing started?"
        ),

        "apple_id_icloud": (
            "Could you tell us what happens when you try to sign in "
            "to your Apple ID, including any error message you see?"
        ),

        "apps_app_store": (
            "Could you tell us which app or App Store issue you're "
            "experiencing and what happens when you try it?"
        ),

        "apple_music": (
            "Could you tell us whether the issue is with playback, "
            "your library, or your Apple Music subscription?"
        ),

        "connectivity": (
            "Could you tell us which connection is affected, such as "
            "Wi-Fi, Bluetooth, or cellular, and which device you're using?"
        ),

        "device_hardware": (
            "Could you tell us which device you're using and describe "
            "what hardware problem you're experiencing?"
        )
    }

    return clarification_questions.get(
        intent,
        (
            "Could you share a few more details about the issue, "
            "including your device model and iOS version?"
        )
    )


print("Final safer response generator created.")

Final safer response generator created.


In [122]:
# Cell 103 — Test Final Safer Response Generator

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    response = generate_support_response(result)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Retrieval similarity: 0.77
Decision: clarify_or_escalate

Final suggested response:
I'd be happy to help. Could you share a few more details about the issue, including your device model and the iOS version 

In [123]:
# Cell 104 — Fix Response Cleanup

def clean_generated_response(text):
    """
    Final cleanup for customer-facing responses.
    """

    text = str(text)

    # Remove leftover Twitter escalation fragments
    patterns = [
        r"\s+and we'll work with you there\.?",
        r"\s+and we’ll work with you there\.?",
        r"\s+and we'll continue there\.?",
        r"\s+and we’ll continue there\.?",
        r"\s+there\.?",
    ]

    for pattern in patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE
        )

    # Fix punctuation spacing
    text = re.sub(r"\?\s+and", "?", text)

    # Remove duplicate spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


print("Final response cleanup created.")

Final response cleanup created.


In [124]:
# Cell 105 — Final End-to-End Test

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    raw_response = generate_support_response(result)
    final_response = clean_generated_response(raw_response)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(final_response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Retrieval similarity: 0.77
Decision: clarify_or_escalate

Final suggested response:
I'd be happy to help. Could you share a few more details about the issue, including your device model and the iOS version 

In [128]:
# Cell 107 — Final Agent Validation

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    raw_response = generate_support_response(result)
    final_response = clean_generated_response(raw_response)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(final_response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Retrieval similarity: 0.77
Decision: clarify_or_escalate

Final suggested response:
I'd be happy to help. Could you share a few more details about the issue, including your device model and the iOS version 

In [130]:
# Cell 109 — Improve Response Generation

def generate_support_response(agent_result):
    """
    Generate a safer customer-facing response.

    Retrieved evidence is used only when it contains meaningful guidance.
    Otherwise, the agent asks an intent-specific clarification question.
    """

    decision = agent_result["decision"]
    intent = agent_result["predicted_intent"]

    clarification_questions = {
        "battery_power": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when you first noticed the battery drain?"
        ),

        "keyboard_input": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and which keyboard or typing issue you're seeing?"
        ),

        "performance_freezing": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when the freezing started?"
        ),

        "apple_id_icloud": (
            "Could you tell us what happens when you try to sign in "
            "to your Apple ID, including any error message you see?"
        ),

        "apps_app_store": (
            "Could you tell us which app or App Store issue you're "
            "experiencing and what happens when you try it?"
        ),

        "apple_music": (
            "Could you tell us whether the issue is with playback, "
            "your library, or your Apple Music subscription?"
        ),

        "connectivity": (
            "Could you tell us which connection is affected, such as "
            "Wi-Fi, Bluetooth, or cellular, and which device you're using?"
        ),

        "device_hardware": (
            "Could you tell us which device you're using and describe "
            "what hardware problem you're experiencing?"
        )
    }

    # -----------------------------------
    # 1. Low-confidence decision
    # -----------------------------------
    if decision == "clarify_or_escalate":
        return clarification_questions.get(
            intent,
            "Could you share a few more details about the issue, including your device model and iOS version?"
        )

    # -----------------------------------
    # 2. Clean retrieved evidence
    # -----------------------------------
    evidence = clean_support_response(
        agent_result["retrieved_support"]
    )

    # -----------------------------------
    # 3. Remove Twitter-specific wording
    # -----------------------------------
    twitter_patterns = [
        r"Let's move over to DM\.?",
        r"DM us and we'll continue there\.?",
        r"Please DM us\.?",
        r"Send us a DM\.?",
        r"Let us know in DM,?",
        r"we'll work with you there\.?",
        r"We want to look into this with you\.?",
        r"We want to help in any way we can\.?"
    ]

    for pattern in twitter_patterns:
        evidence = re.sub(
            pattern,
            "",
            evidence,
            flags=re.IGNORECASE
        )

    evidence = re.sub(r"\s+", " ", evidence).strip()

    # -----------------------------------
    # 4. Detect weak/generic evidence
    # -----------------------------------
    weak_phrases = [
        "which iphone are you using",
        "which device are you using",
        "let us know",
        "we'd be happy to help",
        "we would be happy to help",
        "we're happy to help",
        "we are happy to help",
        "that's unexpected"
    ]

    evidence_lower = evidence.lower()

    is_generic = (
        not evidence
        or len(evidence.split()) < 8
        or any(
            phrase in evidence_lower
            for phrase in weak_phrases
        )
    )

    # -----------------------------------
    # 5. Use evidence only if meaningful
    # -----------------------------------
    if not is_generic:
        return (
            "Thanks for reaching out. "
            + evidence
        )

    # -----------------------------------
    # 6. Retrieved evidence is weak
    #    → ask useful clarification
    # -----------------------------------
    return clarification_questions.get(
        intent,
        "Could you share a few more details about the issue, including your device model and iOS version?"
    )


print("Improved response generator created.")

Improved response generator created.


In [131]:
# Cell 110 — Final Response Validation

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    raw_response = generate_support_response(result)
    final_response = clean_generated_response(raw_response)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(final_response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and when you first noticed the battery drain?
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Retrieval similarity: 0.77
Decision: clarify_or_escalate

Final suggested response:
Could you tell us what happens when you try to sign in to your Apple ID, including any error message you see?
TEST 4
Customer: My iPhone keeps freezing after the latest update.
Intent: 

In [132]:
# Cell 111 — Inspect Retrieved Evidence

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    print("=" * 100)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")

    print("\nRetrieved customer message:")
    print(result["retrieved_customer"])

    print("\nRetrieved support response:")
    print(result["retrieved_support"])

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882

Retrieved customer message:
since I’ve updated to the IOS 11 my battery has been draining to quickly I’ve only been using it for 15 mins& it’s drained

Retrieved support response:
@134147 That's unexpected, but we'd be happy to help. Let's move over to DM. Let us know which iOS version you currently have installed. You can check in Settings &gt; General &gt; About. https://t.co/GDrqU22YpT
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825

Retrieved customer message:
how do I get my keyboard to STOP auto correcting the letter "I" to a symbol. See attached photo.

Retrieved support response:
@128907 We want to look into this with you. DM us and we'll continue there. https://t.co/GDrqU22YpT
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple

In [133]:
# Cell 112 — Better Evidence Quality Detection

def is_useful_evidence(evidence):
    """
    Decide whether retrieved support evidence contains
    actionable guidance rather than only generic escalation language.
    """

    if not evidence:
        return False

    text = evidence.lower().strip()

    # Useful signals in historical support responses
    useful_patterns = [
        # Questions that collect diagnostic information
        r"\bwhich ios version\b",
        r"\bwhich iphone\b",
        r"\bwhich device\b",
        r"\bwhat happens\b",
        r"\bwhat error\b",
        r"\bwhat model\b",

        # Concrete instructions
        r"\bcheck\b",
        r"\bsettings\b",
        r"\bupdate\b",
        r"\brestart\b",
        r"\breset\b",
        r"\binstall\b",
        r"\bbackup\b",
        r"\bfollow\b",

        # Resources
        r"\barticle\b",
        r"\bguide\b",
        r"\bsteps\b",
        r"\bwebsite\b"
    ]

    return any(
        re.search(pattern, text)
        for pattern in useful_patterns
    )


print("Improved evidence quality checker created.")

# Quick sanity check using the five retrieved responses
test_evidence = [
    "That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.",
    "We want to look into this with you.",
    "We can help! Check out this article for guidance with your Apple ID.",
    "We want to help in any way we can. Which iPhone are you using? Let us know in DM.",
    "We're glad to look into this with you! Please DM us. Let us know which iPhone you're using to proceed."
]

for i, evidence in enumerate(test_evidence, 1):
    print(f"Test {i}: {is_useful_evidence(evidence)}")

Improved evidence quality checker created.
Test 1: True
Test 2: False
Test 3: True
Test 4: True
Test 5: True


In [134]:
# Cell 113 — Final Evidence Quality Checker

def is_useful_evidence(evidence):
    """
    Return True only when retrieved evidence contains
    concrete guidance, a resource, or an actionable instruction.

    A generic diagnostic question by itself is not considered
    sufficient evidence.
    """

    if not evidence:
        return False

    text = evidence.lower().strip()

    # Concrete guidance / instructions
    action_patterns = [
        r"\bcheck in settings\b",
        r"\bgo to settings\b",
        r"\bsettings\s*>\s*",
        r"\bupdate\b",
        r"\brestart\b",
        r"\breboot\b",
        r"\breset\b",
        r"\binstall\b",
        r"\bremove\b",
        r"\bturn off\b",
        r"\bturn on\b",
        r"\bfollow the steps\b",
        r"\btry these steps\b",
    ]

    # Useful external resource
    resource_patterns = [
        r"\bcheck out this article\b",
        r"\bthis article\b",
        r"\bguide\b",
        r"\bhelp page\b",
        r"\bsupport page\b",
    ]

    has_action = any(
        re.search(pattern, text)
        for pattern in action_patterns
    )

    has_resource = any(
        re.search(pattern, text)
        for pattern in resource_patterns
    )

    return has_action or has_resource


print("Final evidence quality checker created.")

# Test against the five real retrieved responses

test_evidence = [
    "That's unexpected, but we'd be happy to help. "
    "Let us know which iOS version you currently have installed. "
    "You can check in Settings > General > About.",

    "We want to look into this with you.",

    "We can help! Check out this article for guidance with your Apple ID.",

    "We want to help in any way we can. "
    "Which iPhone are you using? Let us know in DM.",

    "We're glad to look into this with you! "
    "Please DM us. Let us know which iPhone you're using to proceed."
]

for i, evidence in enumerate(test_evidence, 1):
    print(f"Test {i}: {is_useful_evidence(evidence)}")

Final evidence quality checker created.
Test 1: True
Test 2: False
Test 3: True
Test 4: False
Test 5: False


In [135]:
# Cell 114 — Connect Final Evidence Checker

def generate_support_response(agent_result):
    """
    Generate a customer-facing response using retrieved evidence.

    Retrieved evidence is used only when it contains concrete
    guidance or a useful resource. Generic Twitter-style
    escalation responses are replaced with clarification.
    """

    decision = agent_result["decision"]
    intent = agent_result["predicted_intent"]

    clarification_questions = {
        "battery_power": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when you first noticed the battery drain?"
        ),

        "keyboard_input": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and which keyboard or typing issue you're seeing?"
        ),

        "performance_freezing": (
            "Could you tell us which iPhone model and iOS version "
            "you're using, and when the freezing started?"
        ),

        "apple_id_icloud": (
            "Could you tell us what happens when you try to sign in "
            "to your Apple ID, including any error message you see?"
        ),

        "apps_app_store": (
            "Could you tell us which app or App Store issue you're "
            "experiencing and what happens when you try it?"
        ),

        "apple_music": (
            "Could you tell us whether the issue is with playback, "
            "your library, or your Apple Music subscription?"
        ),

        "connectivity": (
            "Could you tell us which connection is affected, such as "
            "Wi-Fi, Bluetooth, or cellular, and which device you're using?"
        ),

        "device_hardware": (
            "Could you tell us which device you're using and describe "
            "what hardware problem you're experiencing?"
        )
    }

    # Low confidence → clarification
    if decision == "clarify_or_escalate":
        return clarification_questions.get(
            intent,
            "Could you share a few more details about the issue, including your device model and iOS version?"
        )

    # Clean retrieved response
    evidence = clean_support_response(
        agent_result["retrieved_support"]
    )

    # Remove Twitter-specific escalation wording
    twitter_patterns = [
        r"Let's move over to DM\.?",
        r"DM us and we'll continue there\.?",
        r"Please DM us\.?",
        r"Send us a DM\.?",
        r"Let us know in DM,?",
        r"we'll work with you there\.?",
        r"We want to look into this with you\.?",
        r"We want to help in any way we can\.?",
        r"We're glad to look into this with you\.?"
    ]

    for pattern in twitter_patterns:
        evidence = re.sub(
            pattern,
            "",
            evidence,
            flags=re.IGNORECASE
        )

    evidence = re.sub(
        r"\s+",
        " ",
        evidence
    ).strip()

    # Use the final evidence-quality checker
    if is_useful_evidence(evidence):
        return "Thanks for reaching out. " + evidence

    # Weak retrieval → clarification
    return clarification_questions.get(
        intent,
        "Could you share a few more details about the issue, including your device model and iOS version?"
    )


print("Response generator connected to final evidence-quality checker.")

Response generator connected to final evidence-quality checker.


In [136]:
 # Cell 115 — Final Agent Test

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    raw_response = generate_support_response(result)
    final_response = clean_generated_response(raw_response)

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Customer: {message}")
    print(f"Intent: {result['predicted_intent']}")
    print(f"Confidence: {result['intent_confidence']}")
    print(f"Retrieval similarity: {result['retrieval_similarity']}")
    print(f"Decision: {result['decision']}")

    print("\nFinal suggested response:")
    print(final_response)

TEST 1
Customer: My battery is draining very quickly after updating to iOS 11.
Intent: battery_power
Confidence: 0.404
Retrieval similarity: 0.882
Decision: use_retrieved_evidence

Final suggested response:
Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About.
TEST 2
Customer: My keyboard keeps changing the word I to something else.
Intent: keyboard_input
Confidence: 0.514
Retrieval similarity: 0.825
Decision: use_retrieved_evidence

Final suggested response:
Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?
TEST 3
Customer: I cannot log in to my Apple ID.
Intent: apple_id_icloud
Confidence: 0.195
Retrieval similarity: 0.77
Decision: clarify_or_escalate

Final suggested response:
Could you tell us what happens when you try to sign in to your Apple ID, including any error message you see?
TEST 4
Custo

In [137]:
from pathlib import Path
import json

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

print("Saving taxonomy validation artifacts...")

Saving taxonomy validation artifacts...


In [138]:
with open(
    output_dir / "evaluation_messages.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_messages,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ evaluation_messages.json saved")

✓ evaluation_messages.json saved


In [141]:
import pandas as pd
from pathlib import Path

evaluation_results = []

for i, message in enumerate(evaluation_messages, 1):

    result = run_support_agent(message)

    raw_response = generate_support_response(result)

    final_response = clean_generated_response(raw_response)

    evaluation_results.append({
        "test_id": i,
        "customer_message": message,
        "predicted_intent": result["predicted_intent"],
        "intent_confidence": result["intent_confidence"],
        "retrieval_similarity": result["retrieval_similarity"],
        "decision": result["decision"],
        "final_response": final_response
    })

evaluation_df = pd.DataFrame(evaluation_results)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_df.to_csv(
    output_dir / "taxonomy_agent_evaluation.csv",
    index=False
)

print("Saved:", output_dir / "taxonomy_agent_evaluation.csv")
print("Rows:", len(evaluation_df))

display(evaluation_df)

Saved: ..\data\processed\taxonomy_agent_evaluation.csv
Rows: 5


,test_id,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision,final_response
0,1,My battery is draining very quickly after updating to iOS 11.,battery_power,0.404,0.882,use_retrieved_evidence,"Thanks for reaching out. That's unexpected, but we'd be happy to help. Let us know which iOS version you currently have installed. You can check in Settings > General > About."
1,2,My keyboard keeps changing the word I to something else.,keyboard_input,0.514,0.825,use_retrieved_evidence,"Could you tell us which iPhone model and iOS version you're using, and which keyboard or typing issue you're seeing?"
2,3,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate,"Could you tell us what happens when you try to sign in to your Apple ID, including any error message you see?"
3,4,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,use_retrieved_evidence,"Could you tell us which iPhone model and iOS version you're using, and when the freezing started?"
4,5,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate,Could you tell us which device you're using and describe what hardware problem you're experiencing?


In [142]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "taxonomy_agent_evaluation.csv"

evaluation_df.to_csv(
    output_file,
    index=False
)

print(f"Saved evaluation results to: {output_file}")
print(f"Rows saved: {len(evaluation_df)}")

Saved evaluation results to: ..\data\processed\taxonomy_agent_evaluation.csv
Rows saved: 5
